# SCAPS-1D optimization of a lead-free FTO/WO$_3$/MASnIBr$_2$/CBTS/Au solar cell — complete workflow

This notebook runs the whole study from the first step to the last, in order:

1. read the solar-cell device from its SCAPS-1D file,
2. check every candidate design with four simple physics rules (the *physics precheck*),
3. optimize ten design variables with three algorithms — PPSR, TPE and a genetic algorithm (GA) — each with six random seeds,
4. compare the three algorithms,
5. characterize the best device (J–V curve, quantum efficiency, band diagram, losses, thickness, temperature and resistance effects, Shockley–Queisser limit),
6. find which design variables matter most,
7. test how robust the best device is to fabrication errors and to uncertain material parameters,
8. train machine-learning models that predict the efficiency and check them carefully.

**Before you start**

* SCAPS-1D 3.3.11 must be installed (Windows). Set `SCAPS_DIR` in step 0.3 to your SCAPS-1D folder.
* Install the Python packages once (step 0.1).
* Run the cells from top to bottom (*Kernel → Restart & Run All*). Every result is written to the folder `output/`
  in the repository; figures go to `output/figures` (PNG 300 dpi, PDF and TIFF 600 dpi, and each panel of a multi-panel figure on its own).
* The complete run takes about 3–4 days of SCAPS-1D time, most of it in step 3. Start with an empty `output/` folder.

The other notebook in this repository, `MASnIBr2_SCAPS_optimization.ipynb`, contains the same analysis together with the
saved outputs of our own run; it can be read without running anything.

| Part | Content | SCAPS-1D time |
|---|---|---|
| 0 | Set-up: packages, imports, configuration | – |
| 1 | Device file and SCAPS-1D connection | 1 min |
| 2 | Physics precheck | – |
| 3 | Optimization with PPSR, TPE and GA (six seeds each) | about 3 days |
| 4 | Comparison of the three algorithms | – |
| 5 | Best device and its characterization | about 8 h |
| 6 | Which variables matter most | – |
| 7 | Robustness: Monte Carlo and material parameters | about 2.5 h |
| 8 | Machine-learning models and their validation | – (about 1 h of CPU time) |
| 9 | Summary | – |

## Part 0 — Set-up

### 0.1 Install the packages
Needed only once. The versions are listed in `requirements.txt`.

In [ ]:
%pip install -r ../requirements.txt

### 0.2 Imports, figure style and figure saving

`use_style()` resets the figure style before each figure (optionally with a few changes), so every figure cell gives the same result on its own.
`save_figure()` writes a figure in three formats and, for a figure with several panels, every panel separately.

In [ ]:
import os, re, math, json, time, shutil, sqlite3, platform, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import optuna
import shap
import sklearn, scipy, xgboost, pvlib
from scipy import stats, integrate, constants
from scipy.optimize import minimize_scalar
from sklearn.model_selection import train_test_split, KFold, GroupKFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

PUB_STYLE = {
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'stix', 'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'axes.linewidth': 0.8, 'lines.linewidth': 1.5, 'lines.markersize': 5,
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8, 'xtick.major.size': 4, 'ytick.major.size': 4,
    'xtick.direction': 'in', 'ytick.direction': 'in', 'xtick.top': True, 'ytick.right': True,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.1,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.grid': False,
}

def use_style(changes=None):
    """Start a figure from the common style, optionally with some style settings changed."""
    plt.rcdefaults()
    plt.rcParams.update(PUB_STYLE)
    plt.rcParams.update(changes or {})

def save_figure(fig, name):
    """Save a figure as PNG (300 dpi), PDF and TIFF (600 dpi) in FIG_DIR, and each panel in FIG_DIR/panels."""
    os.makedirs(FIG_DIR, exist_ok=True)
    base = os.path.join(FIG_DIR, name)
    fig.savefig(base + '.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(base + '.pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(base + '.tif', dpi=600, bbox_inches='tight', facecolor='white',
                pil_kwargs={'compression': 'tiff_lzw'})
    axes = [ax for ax in fig.axes if ax.get_label() != '<colorbar>' and ax.has_data()]
    if len(axes) > 1:
        panel_dir = os.path.join(FIG_DIR, 'panels')
        os.makedirs(panel_dir, exist_ok=True)
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for k, ax in enumerate(axes):
            bb = ax.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted()).expanded(1.03, 1.03)
            fig.savefig(os.path.join(panel_dir, f'{name}_{chr(97 + k)}.png'), dpi=300, bbox_inches=bb)
    print('Figure saved:', base + '.png')

print(f"{'python':<12} {platform.python_version()}  ({platform.system()})")
for m in (np, pd, scipy, sklearn, optuna, shap, xgboost, pvlib, matplotlib):
    print(f'{m.__name__:<12} {m.__version__}')

### 0.3 Configuration

All settings of the study are in this one cell: folders, the SCAPS-1D installation, the device, the search ranges of the
ten design variables, the settings of the three optimizers, and the settings of the later analyses.
Thicknesses are in µm, doping and defect densities in cm$^{-3}$, interface defect densities in cm$^{-2}$, work functions in eV.

In [ ]:
REPO_DIR        = os.path.abspath(os.path.join(os.getcwd(), '..'))
OUT_DIR         = os.path.join(REPO_DIR, 'output')
FIG_DIR         = os.path.join(OUT_DIR, 'figures')
SCAPS_FILES_DIR = os.path.join(REPO_DIR, 'scaps_files')
ORIGINAL_FILE   = os.path.join(SCAPS_FILES_DIR, 'FTO_WO3_MASnIBr2_CBTS_Au.scaps')
TEMPLATE_FILE   = os.path.join(SCAPS_FILES_DIR, 'template_working.scaps')
os.makedirs(FIG_DIR, exist_ok=True)

SCAPS_DIR        = r'D:\Scaps3311'
SCAPS_EXE        = os.path.join(SCAPS_DIR, 'scaps3311.exe')
WORKING_DIR      = SCAPS_DIR
SCRIPT_FILE      = os.path.join(SCAPS_DIR, 'script', 'adaptive_ml.script')
DESTINATION_FILE = os.path.join(SCAPS_DIR, 'def', 'working.scaps')
OUTPUT_IV_FILE   = os.path.join(SCAPS_DIR, 'results', 'out.iv')
OUTPUT_QE_FILE   = os.path.join(SCAPS_DIR, 'results', 'out.qe')
MAX_TIMEOUT      = 120                     # seconds allowed for one SCAPS-1D run

# Device: FTO / WO3 (ETL) / MASnIBr2 (absorber) / CBTS (HTL) / Au
MANUAL_ROLES         = {'FTO': 'TCO', 'WO3': 'ETL', 'MASnIBr2': 'ABSORBER', 'CBTS': 'HTL'}
ABSORBER_BANDGAP_MAP = {'MASnIBr2': 1.75}
MATERIAL_PARAMS = {
    'FTO':      {'chi': 4.40, 'Eg': 3.50},
    'WO3':      {'chi': 3.80, 'Eg': 3.20},
    'MASnIBr2': {'chi': 3.78, 'Eg': 1.75},
    'CBTS':     {'chi': 3.60, 'Eg': 1.90},
}
CBO_RANGE = (-0.10, 0.20)
VBO_RANGE = (-0.10, 0.05)

# Search ranges of the ten design variables
INITIAL_PARAM_RANGES = {
    'TCO':      {},
    'ETL':      {'thickness': (0.020, 0.100), 'ND': (1e16, 1e19), 'defect': (1e13, 1e16)},
    'ABSORBER': {'thickness': (0.30, 1.40), 'NA': None, 'ND': None, 'defect': (1e13, 1e16)},
    'HTL':      {'thickness': (0.030, 0.300), 'NA': (1e17, 1e20), 'defect': (1e13, 1e16)},
}
INTERFACE_RANGE = (1e9, 1e12)

# Fixed values
VALIDATED_FRONT_WF    = 4.002469
VALIDATED_BACK_WF     = 5.10               # Au back contact, not optimized
BACK_CONTACT_WF_RANGE = (5.10, 5.10)
FIXED_ABSORBER_NA     = 1.0e15
FIXED_ABSORBER_ND     = 1.0e15
FIXED_INTERFACE_N     = 1.0e10

# Optimizers
SEEDS                 = [42, 123, 456, 789, 1000, 2024]
N_SEEDS               = len(SEEDS)
TRIALS_PER_ROUND      = 200                # PPSR: TPE trials per round
MAX_ROUNDS            = 5                  # PPSR: at most 5 rounds (1000 trials)
CONVERGENCE_THRESHOLD = 0.05               # PPSR stops when the best PCE changed by < 0.05 percentage points
CONVERGENCE_ROUNDS    = 3                  #   over the last 3 rounds
IMPORTANCE_THRESHOLD  = 0.05
PERCENTILE_LO, PERCENTILE_HI = 10, 90
BUFFER_FACTOR         = 0.20
ROLLBACK_THRESHOLD    = 0.95
JOINT_NARROW_TOP_K    = 20
JOINT_SIGMA_FACTOR    = 2.0
BASELINE_TOTAL_TRIALS = 1000               # TPE trials per seed
GA_TOTAL_TRIALS       = 1000               # GA trials per seed
GA_POP_SIZE, GA_ELITISM, GA_TOURNAMENT_K = 40, 2, 3
GA_MUTATION_RATE, GA_CROSSOVER_RATE, GA_MUTATION_SIGMA = 0.15, 0.80, 0.15
BOOTSTRAP_TOP_K, BOOTSTRAP_N = 10, 2000

# Operating-condition sweeps of the best device
TEMP_SWEEP_POINTS = [250, 260, 270, 280, 290, 300, 310, 320, 330, 340, 350, 360, 370, 380]   # K
RS_SWEEP_POINTS   = [0.05, 0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5,
                     6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 12.0, 15.0]                 # Ohm cm2
RSH_SWEEP_POINTS  = [10, 25, 50, 75, 100, 150, 200, 300, 500, 750, 1000, 1500, 2500,
                     5000, 10000, 50000, 100000]                                               # Ohm cm2

# Monte Carlo fabrication tolerance: relative spread (one standard deviation) of each variable
MC_N_SAMPLES = 500
MC_INPUT_SIGMA = {
    'wo3_thickness':      (0.15, 'lognormal'),
    'absorber_thickness': (0.10, 'lognormal'),
    'cbts_thickness':     (0.15, 'lognormal'),
    'wo3_ND':             (0.30, 'lognormal'),
    'cbts_NA':            (0.30, 'lognormal'),
    'absorber_Nt':        (0.50, 'lognormal'),
    'iface1_Nt':          (1.00, 'lognormal'),
    'iface2_Nt':          (1.00, 'lognormal'),
    'wo3_Nt':             (0.50, 'lognormal'),
    'cbts_Nt':            (0.50, 'lognormal'),
}

# Names of the ten design variables in the result tables
VARS = ['WO3_t', 'MASnIBr2_t', 'CBTS_t', 'WO3_ND', 'CBTS_NA', 'MASnIBr2_defect',
        'WO3_defect', 'CBTS_defect', 'iface_CBTS_MASnIBr2_Nt', 'iface_MASnIBr2_WO3_Nt']
LOG_VARS = VARS[3:]                        # densities: used as log10 values in the analyses

print('Output folder :', OUT_DIR)
print('SCAPS-1D      :', SCAPS_EXE, '| found' if os.path.exists(SCAPS_EXE) else '| NOT FOUND')
print('Seeds         :', SEEDS)

## Part 1 — Device file and SCAPS-1D connection

### 1.1 Load the helper code
`src/scaps_tools.py` holds the longer functions: reading and writing `.scaps` files, running SCAPS-1D, the physics precheck,
the three optimizers, the sweeps and the Monte Carlo run. It is loaded into this notebook so that it uses the configuration above.

In [ ]:
%run -i ../src/scaps_tools.py

### 1.2 Read the initial device
The layers and interfaces are read from `scaps_files/template_working.scaps` (the initial device, PCE 17.80%).

In [ ]:
device = build_device_config(TEMPLATE_FILE)
device.feature_cols = rebuild_feature_cols_from_current_device(device)

for i, l in enumerate(device.layers, 1):
    d = f'{l.thickness_nm:.0f} nm' if l.thickness_nm else '–'
    print(f"{i}. {l.block_name:<10} {l.role:<9} doping {l.doping_type:<5} {d:>8}  {'optimized' if l.optimize else 'fixed'}")
print(f'\nAbsorber {device.absorber_name}, Eg = {device.absorber_eg} eV')
print(f'Conduction-band offset {device.cbo:+.3f} eV, valence-band offset {device.vbo:+.3f} eV')
print('Interfaces:', list(device.interface_space))
print('Design variables:', device.feature_cols)

### 1.3 Test the SCAPS-1D connection
One simulation of the initial device. The expected result is PCE ≈ 17.80%.

In [ ]:
for label, path in [('SCAPS-1D', SCAPS_EXE), ('Initial device', TEMPLATE_FILE)]:
    print(f"{label:<15} {'found' if os.path.exists(path) else 'NOT FOUND'}: {path}")

shutil.copy(TEMPLATE_FILE, DESTINATION_FILE)
write_scaps_script(SCRIPT_FILE, DESTINATION_FILE, OUTPUT_IV_FILE, OUTPUT_QE_FILE)
scaps_runner = ScapsRunner(SCAPS_EXE, WORKING_DIR, SCRIPT_FILE, OUTPUT_IV_FILE, OUTPUT_QE_FILE, timeout=MAX_TIMEOUT)
assert scaps_runner.run(), 'SCAPS-1D did not run. Check SCAPS_DIR in step 0.3.'
Voc, Jsc, FF, eta = parse_iv(OUTPUT_IV_FILE)
print(f'\nInitial device: PCE = {eta:.2f}%  Voc = {Voc:.4f} V  Jsc = {Jsc:.2f} mA/cm2  FF = {FF:.2f}%')
print('Physics check of the result:', physics_postcheck(Voc, Jsc, FF, eta, device))

## Part 2 — Physics precheck

Before a design is simulated, it must pass four rules; otherwise it is rejected without running SCAPS-1D:

| Rule | Condition |
|---|---|
| C1 | absorber thickness ≤ 3 × carrier diffusion length (the diffusion length follows from the absorber defect density) |
| C2 | doping of WO$_3$ and CBTS not above the effective density of states (no degenerate doping) |
| C3 | transport-layer doping above the absorber doping, so that a junction forms |
| C4 | WO$_3$ and CBTS at least 20 nm thick, absorber at least 50 nm |

### 2.1 Self-test of the precheck
Six designed test cases, each of which must be accepted or rejected as expected.

In [ ]:
assert _test_physics_precheck(), 'A precheck test case failed.'
print('All precheck test cases passed.')

## Part 3 — Optimization with PPSR, TPE and GA

The ten design variables are optimized for the highest PCE with three algorithms, each with the six seeds of step 0.3:

* **PPSR** (progressive parameter-space refinement) runs the TPE sampler in rounds of 200 trials. After each round it narrows
  the ranges of the variables that SHAP finds most important, and it stops when the best PCE no longer improves.
* **TPE** (tree-structured Parzen estimator) alone, with 1000 trials per seed — the baseline.
* **GA**, a genetic algorithm with 1000 trials per seed.

Every trial is stored in an SQLite database in `output/` (`trials.db`, `baseline_trials.db` and `ga_trials.db`).

### 3.1 PPSR (about 16 h)

In [ ]:
ppsr_results = run_experiment()

### 3.2 TPE (about 24 h)

In [ ]:
tpe_results = run_baseline_experiment()

### 3.3 GA (about 27 h)

In [ ]:
ga_results = run_ga_experiment()

## Part 4 — Comparison of the three algorithms

### 4.1 Load all trials
`TRIALS` holds every trial of an algorithm (also the rejected and failed ones), `VALID` only the successfully simulated trials,
and `BEST_PER_SEED` the best PCE that each algorithm found with each seed.

In [ ]:
def load_trials(db_file):
    with sqlite3.connect(os.path.join(OUT_DIR, db_file)) as con:
        return pd.read_sql('SELECT * FROM trials ORDER BY id', con)

ALGOS  = ['PPSR', 'TPE', 'GA']
TRIALS = {'PPSR': load_trials('trials.db'), 'TPE': load_trials('baseline_trials.db'), 'GA': load_trials('ga_trials.db')}
VALID  = {a: df[df.valid_run == 1].reset_index(drop=True) for a, df in TRIALS.items()}
BEST_PER_SEED = pd.DataFrame({a: VALID[a].groupby('seed').eta.max() for a in ALGOS})

CMP_DIR = os.path.join(OUT_DIR, 'optimizer_comparison')
os.makedirs(CMP_DIR, exist_ok=True)
BEST_PER_SEED.to_csv(os.path.join(CMP_DIR, 'best_pce_per_seed.csv'))

for a in ALGOS:
    print(f'{a:<5} {len(TRIALS[a]):5d} trials, {len(VALID[a]):5d} simulated successfully')
print('\nBest PCE (%) per seed:')
print(BEST_PER_SEED.round(4).to_string())

### 4.2 Statistical comparison of the final PCE
Mean and sample standard deviation over the six seeds, one-way ANOVA, and pairwise Welch t-tests and Mann–Whitney U tests
(p-values Bonferroni-corrected for three comparisons; Cohen's d as effect size).

In [ ]:
champs = {a: BEST_PER_SEED[a].values for a in ALGOS}

desc = pd.DataFrame({a: {'mean': v.mean(), 'sd': v.std(ddof=1), 'min': v.min(), 'max': v.max(), 'n': len(v)}
                     for a, v in champs.items()}).T
F_stat, p_anova = stats.f_oneway(*champs.values())

rows = []
for a, b in [('PPSR', 'TPE'), ('PPSR', 'GA'), ('TPE', 'GA')]:
    x, y = champs[a], champs[b]
    t, p_t = stats.ttest_ind(x, y, equal_var=False)
    U, p_u = stats.mannwhitneyu(x, y, alternative='two-sided')
    rows.append({'pair': f'{a} vs {b}', 'mean_difference_pp': x.mean() - y.mean(),
                 'welch_t': t, 'welch_p_bonferroni': min(3 * p_t, 1.0),
                 'mann_whitney_U': U, 'mann_whitney_p_bonferroni': min(3 * p_u, 1.0),
                 'cohens_d': (x.mean() - y.mean()) / np.sqrt((x.var(ddof=1) + y.var(ddof=1)) / 2)})
pairwise = pd.DataFrame(rows)

print(desc.round(4).to_string())
print(f'\nOne-way ANOVA: F = {F_stat:.3f}, p = {p_anova:.2e}\n')
print(pairwise.round(4).to_string(index=False))

with open(os.path.join(CMP_DIR, 'statistics.json'), 'w') as f:
    json.dump({'descriptive': desc.to_dict(orient='index'),
               'anova': {'F': float(F_stat), 'p': float(p_anova)},
               'pairwise': pairwise.to_dict(orient='records'),
               'best_pce_per_seed': {a: v.tolist() for a, v in champs.items()}}, f, indent=2)

### 4.3 Convergence: best PCE versus trial number
(a) Best PCE found so far versus trial number: line = median of the six seeds, band = range over the six seeds.
Rejected and failed trials count as trials but do not improve the best PCE.
(b) Number of trials needed to reach five absolute PCE targets: bar = median, error bar = range;
"k/6" means that only k of the six seeds reached the target.

In [ ]:
use_style({'font.weight': 'bold', 'axes.labelweight': 'bold', 'font.size': 11, 'axes.linewidth': 1.2,
           'xtick.major.width': 1.1, 'ytick.major.width': 1.1, 'xtick.minor.visible': True,
           'ytick.minor.visible': True, 'legend.edgecolor': 'black', 'legend.fancybox': False,
           'mathtext.default': 'regular'})
COLORS  = {'PPSR': '#1f5fbf', 'TPE': '#d62728', 'GA': '#2a9d3a'}
LINES   = {'PPSR': '-', 'TPE': '--', 'GA': '-.'}
TARGETS = [25.00, 25.20, 25.24, 25.25, 25.26]

def running_best(df):
    """Best PCE so far for every seed (rows = seeds, columns = trial number)."""
    curves = []
    for _, g in df.groupby('seed'):
        eta = np.where(g.valid_run == 1, g.eta, np.nan)
        curves.append(np.fmax.accumulate(np.nan_to_num(eta, nan=-1)))
    M = np.full((len(curves), max(len(c) for c in curves)), np.nan)
    for i, c in enumerate(curves):
        M[i, :len(c)] = c
    return M

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.9))
for a in ALGOS:
    M = running_best(TRIALS[a])
    x = np.arange(1, M.shape[1] + 1)
    ax1.fill_between(x, np.nanmin(M, 0), np.nanmax(M, 0), color=COLORS[a], alpha=0.15, lw=0)
    ax1.plot(x, np.nanmedian(M, 0), LINES[a], color=COLORS[a], lw=1.8, label=a)
ax1.axvline(200, color='grey', lw=1, ls=':')
ax1.text(212, 24.93, 'PPSR round 1\n= TPE', fontsize=9, va='bottom')
ax1.set_xlim(0, 1000); ax1.set_ylim(24.90, 25.30)
ax1.set_xlabel('Trial number'); ax1.set_ylabel('Best-achieved PCE (%)')
ax1.legend(loc='lower right', fontsize=9.5)

w = 0.26
for k, a in enumerate(ALGOS):
    M = running_best(TRIALS[a])
    med, lo, hi, n_reached = [], [], [], []
    for t in TARGETS:
        n_trials = np.array([np.argmax(r >= t) + 1 if (r >= t).any() else np.nan for r in M], float)
        med.append(np.nanmedian(n_trials)); lo.append(np.nanmin(n_trials)); hi.append(np.nanmax(n_trials))
        n_reached.append(int(np.isfinite(n_trials).sum()))
    xs, med = np.arange(len(TARGETS)) + (k - 1) * w, np.array(med)
    ax2.bar(xs, med, w, color=COLORS[a], edgecolor='black', lw=0.8, label=a,
            yerr=[med - np.array(lo), np.array(hi) - med], capsize=3, error_kw=dict(lw=1, ecolor='black'))
    for xi, h, c in zip(xs, hi, n_reached):
        if c < 6:
            ax2.text(xi, h + 18, f'{c}/6', ha='center', fontsize=8.5)
    print(f'{a:<5} median trials to {TARGETS}: {med}; seeds that reached each target: {n_reached}')
ax2.set_xticks(range(len(TARGETS))); ax2.set_xticklabels([f'{t:.2f}%' for t in TARGETS])
ax2.tick_params(axis='x', which='minor', bottom=False, top=False)
ax2.set_xlabel('Absolute PCE target'); ax2.set_ylabel('Trials required to reach target')
ax2.set_ylim(0, 1050); ax2.legend(loc='upper left', fontsize=9.5)

fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.text(0.27, 0.015, '(a)', ha='center', fontsize=13)
fig.text(0.765, 0.015, '(b)', ha='center', fontsize=13)
save_figure(fig, 'convergence')
plt.show()

### 4.4 PPSR and TPE seed by seed
Both algorithms use the same six seeds, so their best PCE values can be compared in pairs: paired t-test, Wilcoxon
signed-rank test, mean difference with its 95% confidence interval, and Cohen's d$_z$.

In [ ]:
pair = BEST_PER_SEED[['PPSR', 'TPE']].copy()
pair['difference_pp'] = pair['PPSR'] - pair['TPE']
d, n = pair['difference_pp'].values, len(pair)

paired_t = stats.ttest_rel(pair['PPSR'], pair['TPE'])
wilcoxon = stats.wilcoxon(pair['PPSR'], pair['TPE'], alternative='two-sided')
d_mean, d_sd = d.mean(), d.std(ddof=1)
half_ci = stats.t.ppf(0.975, n - 1) * d_sd / np.sqrt(n)
ci_lo, ci_hi = d_mean - half_ci, d_mean + half_ci

print(pair.round(4).to_string())
print(f"\nPPSR {pair['PPSR'].mean():.4f} ± {pair['PPSR'].std(ddof=1):.4f}%   "
      f"TPE {pair['TPE'].mean():.4f} ± {pair['TPE'].std(ddof=1):.4f}%")
print(f'Mean difference {d_mean:+.4f} percentage points, 95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}]')
print(f'Paired t-test: t = {paired_t.statistic:.3f}, p = {paired_t.pvalue:.4f}')
print(f'Wilcoxon signed-rank: W = {wilcoxon.statistic:.1f}, p = {wilcoxon.pvalue:.4f}')
print(f"Cohen's dz = {d_mean / d_sd:+.3f}")

pair.to_csv(os.path.join(CMP_DIR, 'ppsr_vs_tpe_per_seed.csv'))
with open(os.path.join(CMP_DIR, 'ppsr_vs_tpe_paired.json'), 'w') as f:
    json.dump({'mean_difference_pp': d_mean, 'ci95_pp': [ci_lo, ci_hi],
               'paired_t': {'t': paired_t.statistic, 'p': paired_t.pvalue},
               'wilcoxon': {'W': float(wilcoxon.statistic), 'p': float(wilcoxon.pvalue)},
               'cohens_dz': d_mean / d_sd}, f, indent=2)

use_style({'font.size': 11, 'axes.labelsize': 12, 'axes.labelweight': 'bold', 'axes.linewidth': 1.2,
           'xtick.labelsize': 10, 'ytick.labelsize': 10, 'xtick.major.size': 5, 'ytick.major.size': 5,
           'xtick.major.width': 1.1, 'ytick.major.width': 1.1})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.8, 4.2), gridspec_kw={'width_ratios': [1.1, 1.0], 'wspace': 0.34})
for i in range(n):
    ax1.plot([0, 1], [pair['TPE'].iloc[i], pair['PPSR'].iloc[i]], color='gray', lw=1.1, alpha=0.65, zorder=1)
for xpos, a, c, xm in [(0, 'TPE', '#d62728', -0.10), (1, 'PPSR', '#1f77b4', 1.10)]:
    ax1.scatter(np.full(n, xpos), pair[a], s=55, color=c, edgecolor='black', lw=0.6, zorder=3)
    ax1.errorbar(xm, pair[a].mean(), yerr=pair[a].std(ddof=1), fmt='s', color=c, ecolor='black',
                 elinewidth=1.1, capsize=4, markersize=7, zorder=4)
ax1.set_xlim(-0.35, 1.35); ax1.set_xticks([0, 1]); ax1.set_xticklabels(['TPE', 'PPSR'])
ax1.set_ylabel('Optimized PCE (%)')
y_all = pair[['TPE', 'PPSR']].values.ravel()
pad = max(0.002, 0.18 * (y_all.max() - y_all.min()))
ax1.set_ylim(y_all.min() - pad, y_all.max() + pad)
ax1.text(0.5, 0.96, f'Paired t-test: p = {paired_t.pvalue:.4f}', transform=ax1.transAxes,
         ha='center', va='top', fontsize=10, fontweight='bold')

ax2.axhline(0, color='black', lw=1.0)
ax2.bar(np.arange(n), d, color='#444444', edgecolor='black', lw=0.7, alpha=0.85)
ax2.axhline(d_mean, color='#ff7f0e', ls='--', lw=1.5, label=f'Mean Δ = {d_mean:+.4f} pp')
ax2.fill_between([-0.5, n - 0.5], ci_lo, ci_hi, color='#ff7f0e', alpha=0.15, label='95% CI')
ax2.set_xlim(-0.5, n - 0.5); ax2.set_xticks(np.arange(n)); ax2.set_xticklabels(pair.index.astype(str))
ax2.set_xlabel('Random seed'); ax2.set_ylabel('PCE difference, PPSR − TPE (pp)')
ax2.legend(loc='best', fontsize=8)
for ax in (ax1, ax2):
    ax.grid(axis='y', ls='--', lw=0.5, alpha=0.35); ax.set_axisbelow(True)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight('bold')

plt.subplots_adjust(left=0.085, right=0.985, top=0.95, bottom=0.23, wspace=0.34)
for ax, title in [(ax1, '(a) Paired final-PCE comparison'), (ax2, '(b) Per-seed PCE difference')]:
    pos = ax.get_position()
    fig.text(pos.x0 + pos.width / 2, 0.085, title, ha='center', va='center', fontsize=11, fontweight='bold')
save_figure(fig, 'ppsr_vs_tpe_per_seed')
plt.show()

### 4.5 Cost of the three algorithms
* The first PPSR round is a plain TPE run, so it must give exactly the same 200 trials as TPE with the same seed.
* How many trials were rejected by the precheck (a trial can break more than one rule; it is counted under C2 if it breaks C2),
  how many SCAPS-1D runs failed, and the total run time.
* The best PCE of each algorithm after the same budget of 600 trials (the budget at which PPSR stopped).

In [ ]:
print('First PPSR round compared with TPE (same seed):')
for s in SEEDS:
    p = TRIALS['PPSR'][TRIALS['PPSR'].seed == s].reset_index(drop=True)
    t = TRIALS['TPE'][TRIALS['TPE'].seed == s].reset_index(drop=True)
    same = [np.allclose(p.loc[i, VARS].astype(float), t.loc[i, VARS].astype(float), rtol=1e-9, equal_nan=True)
            for i in range(min(len(p), len(t)))]
    first_diff = same.index(False) if False in same else None
    print(f'  seed {s:<5} identical trials: {sum(same):<4} first different trial: {first_diff}')

print('\nTrials, precheck rejections, failures and run time:')
hours, best600 = {}, {}
for a in ALGOS:
    df = TRIALS[a]
    reason = df.invalid_reason.fillna('')
    pre = reason.str.startswith('Pre-check')
    n_c2 = int((pre & reason.str.contains('Nv')).sum())
    n_c1 = int((pre & reason.str.contains('L_diff') & ~reason.str.contains('Nv')).sum())
    failed = int(((df.scaps_run_success == 0) & ~pre).sum())
    hours[a] = df.runtime_seconds.fillna(0).sum() / 3600
    best600[a] = [g.iloc[:600].loc[g.iloc[:600].valid_run == 1, 'eta'].max() for _, g in df.groupby('seed')]
    print(f'  {a:<5} {len(df):5d} trials | rejected {pre.sum():4d} (C2 doping limit {n_c2}, otherwise C1 thickness limit {n_c1}) '
          f'| SCAPS-1D failures {failed} | {hours[a]:5.1f} h')
print(f"\nPPSR needed {(1 - hours['PPSR'] / hours['TPE']) * 100:.0f}% less time than TPE "
      f"and {(1 - hours['PPSR'] / hours['GA']) * 100:.0f}% less than GA.")

print('\nBest PCE after 600 trials (mean ± SD over six seeds):')
for a in ALGOS:
    print(f'  {a:<5} {np.mean(best600[a]):.4f} ± {np.std(best600[a], ddof=1):.4f}%')

## Part 5 — Best device and its characterization

### 5.1 Select the final device and simulate it
The final device is the best trial of all 18 runs (three algorithms × six seeds). Its `.scaps` file is written and simulated
once more with all SCAPS-1D outputs: J–V curve, quantum efficiency, energy bands, generation and recombination.

Two small helpers are defined here and used in the rest of the notebook:
`device_updates()` gives the layer and interface values of the final device (optionally with other thicknesses), and
`run_scaps()` simulates one `.scaps` file and returns (Voc, Jsc, FF, PCE).

In [ ]:
FINAL_DIR = os.path.join(OUT_DIR, 'final_device')
os.makedirs(FINAL_DIR, exist_ok=True)

all_valid = pd.concat([VALID[a].assign(algorithm=a) for a in ALGOS], ignore_index=True)
best = all_valid.loc[all_valid.eta.idxmax()]
FINAL = {'algorithm': best.algorithm, 'seed': int(best.seed), 'trial_number': int(best.trial_number),
         'PCE': float(best.eta), 'Voc': float(best.Voc), 'Jsc': float(best.Jsc), 'FF': float(best.FF),
         **{v: float(best[v]) for v in VARS}, 'back_contact_wf': float(best.back_contact_wf)}

def device_updates(d, absorber_um=None, etl_um=None, htl_um=None):
    """Layer and interface values of device d for build_updated_scaps_file; thicknesses (µm) can be replaced."""
    values = {'ETL':      (etl_um or d['WO3_t'], None, d['WO3_ND'], d['WO3_defect']),
              'HTL':      (htl_um or d['CBTS_t'], d['CBTS_NA'], None, d['CBTS_defect']),
              'ABSORBER': (absorber_um or d['MASnIBr2_t'], None, None, d['MASnIBr2_defect'])}
    layers = {}
    for layer in device.layers:
        if layer.role.upper() in values:
            t, na, nd, nt = values[layer.role.upper()]
            layers[layer.block_name] = {'thickness_um': t, 'doping_type': None, 'na_val': na, 'nd_val': nd, 'defect': nt}
    ifaces = {k: d[k] for k in ('iface_CBTS_MASnIBr2_Nt', 'iface_MASnIBr2_WO3_Nt')}
    return layers, ifaces

def build_device(path, layers, ifaces, verify=False):
    build_updated_scaps_file(TEMPLATE_FILE, path, layer_updates=layers, iface_vals=ifaces,
                             back_contact_wf=FINAL['back_contact_wf'], verify=verify)

def run_scaps(scaps_file):
    """Simulate one .scaps file; returns (Voc, Jsc, FF, PCE), or None if SCAPS-1D fails."""
    write_scaps_script(SCRIPT_FILE, scaps_file, OUTPUT_IV_FILE, OUTPUT_QE_FILE)
    if not scaps_runner.run():
        return None
    return parse_iv(OUTPUT_IV_FILE)

FINAL_SCAPS = os.path.join(FINAL_DIR, 'final_device.scaps')
build_device(FINAL_SCAPS, *device_updates(FINAL), verify=True)

FINAL_FILES = {ext: os.path.join(FINAL_DIR, f'final_device.{ext}') for ext in ('iv', 'qe', 'eb', 'gen', 'rec')}
write_scaps_script(SCRIPT_FILE, FINAL_SCAPS, FINAL_FILES['iv'], FINAL_FILES['qe'], run_qe=True,
                   eb_file=FINAL_FILES['eb'], gen_file=FINAL_FILES['gen'], rec_file=FINAL_FILES['rec'])
assert ScapsRunner(SCAPS_EXE, WORKING_DIR, SCRIPT_FILE, FINAL_FILES['iv'], FINAL_FILES['qe'],
                   timeout=MAX_TIMEOUT).run(), 'SCAPS-1D run of the final device failed.'

with open(os.path.join(FINAL_DIR, 'final_device.json'), 'w') as f:
    json.dump(FINAL, f, indent=2)

print(f"Final device: {FINAL['algorithm']}, seed {FINAL['seed']}, trial {FINAL['trial_number']}")
print(f"PCE = {FINAL['PCE']:.4f}%  Voc = {FINAL['Voc']:.4f} V  Jsc = {FINAL['Jsc']:.4f} mA/cm2  FF = {FINAL['FF']:.4f}%")
print(f"Thickness WO3 / MASnIBr2 / CBTS = {FINAL['WO3_t']*1000:.1f} / {FINAL['MASnIBr2_t']*1000:.1f} / "
      f"{FINAL['CBTS_t']*1000:.1f} nm")
for v in VARS[3:]:
    print(f'  {v:<24} {FINAL[v]:.3e}')

### 5.2 J–V curve and quantum efficiency

In [ ]:
def read_two_columns(path):
    """All lines of a SCAPS-1D output file that start with two numbers."""
    rows = []
    for line in open(path, encoding='latin-1'):
        parts = [p for p in line.strip().split('\t') if p.strip()]
        try:
            rows.append((float(parts[0]), float(parts[1])))
        except (ValueError, IndexError):
            continue
    return np.array(rows)

jv, qe = read_two_columns(FINAL_FILES['iv']), read_two_columns(FINAL_FILES['qe'])
pd.DataFrame(jv, columns=['V (V)', 'J (mA/cm2)']).to_csv(os.path.join(FINAL_DIR, 'jv_data.csv'), index=False)
pd.DataFrame(qe, columns=['Wavelength (nm)', 'QE (%)']).to_csv(os.path.join(FINAL_DIR, 'qe_data.csv'), index=False)

use_style()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.8))
ax1.plot(jv[:, 0], jv[:, 1], color='#1f77b4', lw=2.0)
ax1.axhline(0, color='gray', lw=0.5, ls=':'); ax1.axvline(0, color='gray', lw=0.5, ls=':')
ax1.set_xlabel('Voltage (V)'); ax1.set_ylabel('Current density (mA/cm²)')
ax1.set_title(f"(a) J-V curve  (PCE = {FINAL['PCE']:.2f}%)", fontsize=11, fontweight='bold')
ax1.text(0.05, 0.05, f"$V_{{oc}}$ = {FINAL['Voc']:.4f} V\n$J_{{sc}}$ = {FINAL['Jsc']:.2f} mA/cm²\nFF = {FINAL['FF']:.2f}%",
         transform=ax1.transAxes, va='bottom', ha='left', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.9))
ax2.plot(qe[:, 0], qe[:, 1], color='#2ca02c', lw=2.0)
ax2.set_xlabel('Wavelength (nm)'); ax2.set_ylabel('External quantum efficiency (%)')
ax2.set_title('(b) External quantum efficiency', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 105)
plt.tight_layout()
save_figure(fig, 'jv_and_qe')
plt.show()

### 5.3 Energy band diagram (illuminated, 0 V)

In [ ]:
lines = open(FINAL_FILES['eb'], encoding='latin-1').readlines()
start = next(k for k, ln in enumerate(lines) if ln.split() and ln.split()[0] == 'i' and 'x(um)' in ln)
cols = [c.strip() for c in lines[start].strip().split('\t') if c.strip()]
rows = []
for ln in lines[start + 1:]:
    parts = ln.split()
    if not parts:
        if rows:
            break
        continue
    try:
        rows.append([float(p) for p in parts[:len(cols)]])
    except ValueError:
        break
bands = pd.DataFrame(rows, columns=cols[:len(rows[0])])
bands.to_csv(os.path.join(FINAL_DIR, 'band_data_0V.csv'), index=False)

use_style()
fig, ax = plt.subplots(figsize=(7.2, 4.2))
x_nm = bands['x(um)'] * 1000
ax.plot(x_nm, bands['Ec(eV)'], label='$E_C$', color='#1f77b4', lw=2)
ax.plot(x_nm, bands['Ev(eV)'], label='$E_V$', color='#ff7f0e', lw=2)
ax.plot(x_nm, bands['Fn(eV)'], label='$E_{Fn}$', color='#2ca02c', ls='--', lw=1.5)
ax.plot(x_nm, bands['Fp(eV)'], label='$E_{Fp}$', color='#d62728', ls='--', lw=1.5)
ax.set_xlabel('Position (nm)'); ax.set_ylabel('Energy (eV)')
ax.legend(loc='center right')
plt.tight_layout()
save_figure(fig, 'energy_bands_0V')
plt.show()

### 5.4 Where the current is lost
From the full J–V table of SCAPS-1D: the generated and the collected current at 0 V, and the recombination current at
the open-circuit voltage split into bulk and interface parts and into SRH, radiative and Auger recombination.

In [ ]:
def read_iv_table(path):
    """The J-V table of a SCAPS-1D .iv file: voltage, total current and its generation and recombination parts."""
    lines = open(path, encoding='latin-1').readlines()
    start = next((i for i, ln in enumerate(lines) if 'v(V)' in ln and 'jtot' in ln), None)
    if start is None:
        return None
    cols = [c.strip() for c in lines[start].split('\t') if c.strip()]
    rows = []
    for ln in lines[start + 1:]:
        parts = [p.strip() for p in ln.strip().split('\t') if p.strip()]
        try:
            vals = [float(p) for p in parts]
        except ValueError:
            break
        if len(vals) >= len(cols):
            rows.append(vals[:len(cols)])
    return pd.DataFrame(rows, columns=cols) if len(rows) >= 5 else None

def current_at(table, column, V):
    return abs(float(np.interp(V, table['v(V)'], table[column])))

iv_table = read_iv_table(FINAL_FILES['iv'])
iv_table.to_csv(os.path.join(FINAL_DIR, 'jv_full_table.csv'), index=False)

LOSSES = {'Jsc': current_at(iv_table, 'jtot(mA/cm2)', 0.0),
          'J_generated': current_at(iv_table, 'j_total_gen(mA/cm2)', 0.0)}
LOSSES['collected_share_pct'] = 100 * LOSSES['Jsc'] / LOSSES['J_generated']
for name, col in [('total', 'j_total_rec'), ('bulk', 'jbulk'), ('interface', 'jifr'),
                  ('SRH', 'j_SRH'), ('radiative', 'j_Radiative'), ('Auger', 'j_Auger')]:
    LOSSES[f'J_rec_{name}_at_Voc'] = current_at(iv_table, f'{col}(mA/cm2)', FINAL['Voc'])

with open(os.path.join(FINAL_DIR, 'loss_decomposition.json'), 'w') as f:
    json.dump(LOSSES, f, indent=2)

print(f"Generated current {LOSSES['J_generated']:.4f} mA/cm2, collected (Jsc) {LOSSES['Jsc']:.4f} mA/cm2 "
      f"({LOSSES['collected_share_pct']:.2f}%)")
total = LOSSES['J_rec_total_at_Voc']
print(f'Recombination current at Voc: {total:.4f} mA/cm2')
for name in ('bulk', 'interface', 'SRH', 'radiative', 'Auger'):
    j = LOSSES[f'J_rec_{name}_at_Voc']
    print(f'  {name:<10} {j:.4f} mA/cm2  ({100 * j / total:5.1f}%)')

### 5.5 Thickness of one layer at a time
Each layer thickness is varied while all other variables keep their final values (35 SCAPS-1D runs, a few minutes).

In [ ]:
SWEEP_DIR = os.path.join(OUT_DIR, 'thickness_sweeps')
os.makedirs(SWEEP_DIR, exist_ok=True)
THICKNESS_SWEEPS = {
    'absorber': list(range(200, 1401, 100)),
    'etl':      [20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 100],
    'htl':      [30, 50, 70, 90, 120, 150, 180, 210, 240, 270, 300],
}

for layer, values in THICKNESS_SWEEPS.items():
    rows = []
    for nm in values:
        path = os.path.join(SWEEP_DIR, f'{layer}_{nm}nm.scaps')
        build_device(path, *device_updates(FINAL, **{f'{layer}_um': nm / 1000}))
        r = run_scaps(path)
        rows.append({f'{layer}_nm': nm, 'PCE': r[3], 'Voc': r[0], 'Jsc': r[1], 'FF': r[2]} if r else
                    {f'{layer}_nm': nm, 'PCE': None})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(SWEEP_DIR, f'sweep_{layer}.csv'), index=False)
    top = df.loc[df.PCE.idxmax()]
    print(f'{layer:<8} best at {top[f"{layer}_nm"]:.0f} nm: PCE {top.PCE:.4f}%')

### 5.6 Temperature, series resistance and shunt resistance
The final device is simulated at 14 temperatures (250–380 K), 26 series resistances and 17 shunt resistances (about 1 h).

In [ ]:
sweeps = run_parameter_sweeps(FINAL_SCAPS, 'operating_conditions')

### 5.7 Figures of the temperature and resistance sweeps

In [ ]:
from matplotlib.ticker import LogLocator, NullFormatter

sweep = json.load(open(os.path.join(OUT_DIR, 'operating_conditions', 'sweep_analysis.json')))

def xy(points, key, column='eta'):
    x = np.array([p[key] for p in points], float)
    y = np.array([p[column] for p in points], float)
    order = np.argsort(x)
    return x[order], y[order]

T, T_pce = xy(sweep['temperature_points'], 'T_K')
_, T_voc = xy(sweep['temperature_points'], 'T_K', 'Voc')
Rs, Rs_pce = xy(sweep['rs_points'], 'Rs')
Rsh, Rsh_pce = xy(sweep['rsh_points'], 'Rsh')
ok = np.isfinite(Rsh) & np.isfinite(Rsh_pce) & (Rsh > 0) & (Rsh_pce > 0) & (Rsh_pce < 35)
Rsh, Rsh_pce = Rsh[ok], Rsh_pce[ok]
print(f'PCE at 300 K: {T_pce[np.argmin(abs(T - 300))]:.3f}%')
print(f'Voc from {T.min():.0f} K to {T.max():.0f} K: {T_voc[0]:.3f} V to {T_voc[-1]:.3f} V')

use_style({'font.size': 10, 'axes.labelsize': 11, 'axes.labelweight': 'bold', 'axes.linewidth': 1.25,
           'xtick.major.size': 5, 'ytick.major.size': 5, 'xtick.major.width': 1.1, 'ytick.major.width': 1.1,
           'xtick.minor.size': 3, 'ytick.minor.size': 3, 'xtick.minor.width': 0.9, 'ytick.minor.width': 0.9})

def sweep_panel(x, y, color, xlabel, name, log_x=False, mark_300K=False):
    fig = plt.figure(figsize=(3.2, 3.35))
    ax = fig.add_axes([0.24, 0.19, 0.731, 0.761])
    (ax.semilogx if log_x else ax.plot)(x, y, '-', color=color, lw=1.8, zorder=2)
    ax.scatter(x, y, s=22, color=color, edgecolor='black', lw=0.35, zorder=3)
    pad = 0.08 * (y.max() - y.min())
    ax.set_ylim(max(0, y.min() - pad) if log_x else y.min() - pad, y.max() + pad)
    if log_x:
        ax.set_xlim(x.min() * 0.85, x.max() * 1.15)
        ax.xaxis.set_major_locator(LogLocator(base=10.0))
        ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1))
        ax.xaxis.set_minor_formatter(NullFormatter())
    else:
        ax.set_xlim(x.min() - 5, x.max() + 5)
    if mark_300K:
        ax.axvline(300, color='gray', ls=':', lw=1.2, alpha=0.8)
        ax.text(302, y.min() + 0.05 * (y.max() - y.min()), '300 K', fontsize=8.5, fontweight='bold',
                color='dimgray', rotation=90, va='bottom')
    ax.set_xlabel(xlabel); ax.set_ylabel('PCE (%)')
    ax.grid(axis='y', ls='--', lw=0.5, alpha=0.35); ax.set_axisbelow(True)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight('bold')
    save_figure(fig, name)
    plt.show()

sweep_panel(T, T_pce, '#1f77b4', 'Temperature (K)', 'pce_vs_temperature', mark_300K=True)
sweep_panel(Rs, Rs_pce, '#ff7f0e', r'Series resistance, $R_s$ ($\Omega$ cm$^2$)', 'pce_vs_series_resistance', log_x=True)
sweep_panel(Rsh, Rsh_pce, '#2ca02c', r'Shunt resistance, $R_{sh}$ ($\Omega$ cm$^2$)', 'pce_vs_shunt_resistance', log_x=True)

### 5.8 Shockley–Queisser limit
The detailed-balance (Shockley–Queisser) limit is calculated from the standard AM1.5G spectrum (ASTM G173-03, from `pvlib`)
at 300 K. As a check, the maximum over all bandgaps must be close to the published value (about 33.7% at 1.34 eV).
The final device is then compared with the limit at the absorber bandgap, 1.75 eV.

In [ ]:
h, c, kB, q = constants.Planck, constants.speed_of_light, constants.Boltzmann, constants.elementary_charge
T_CELL = 300.0
spectrum = pvlib.spectrum.get_reference_spectra(standard='ASTM G173-03')
WL, AM15G = spectrum.index.values, spectrum['global'].values
P_IN = float(np.trapezoid(AM15G, WL))

def sq_limit(Eg):
    """Shockley-Queisser limit of a single-junction cell with bandgap Eg (eV)."""
    m = WL <= 1239.84 / Eg
    Jsc = q * np.trapezoid(AM15G[m] / (h * c / (WL[m] * 1e-9)), WL[m])
    def blackbody(E):
        arg = E * q / (kB * T_CELL)
        return 0.0 if arg > 700 else (2 * np.pi / (h**3 * c**2)) * (E * q)**2 / np.expm1(arg) * q
    J0 = q * integrate.quad(blackbody, Eg, 6.0, limit=300, epsrel=1e-6)[0]
    Voc = (kB * T_CELL / q) * np.log(Jsc / J0 + 1)
    def neg_power(V):
        return 0 if V <= 0 or V >= Voc else -(Jsc - J0 * np.expm1(q * V / (kB * T_CELL))) * V
    V_mpp = minimize_scalar(neg_power, bounds=(0.01, Voc * 0.999), method='bounded', options={'xatol': 1e-7}).x
    P_mpp = (Jsc - J0 * np.expm1(q * V_mpp / (kB * T_CELL))) * V_mpp
    return {'Eg_eV': Eg, 'PCE_pct': P_mpp / P_IN * 100, 'Voc_V': Voc, 'Jsc_mA_cm2': Jsc / 10,
            'FF_pct': P_mpp / (Jsc * Voc) * 100}

EG = 1.75
SQ = sq_limit(EG)
EG_SCAN = np.arange(0.5, 3.0001, 0.002)
PCE_SCAN = np.array([sq_limit(e)['PCE_pct'] for e in EG_SCAN])
SHARE_OF_SQ = {'PCE': FINAL['PCE'] / SQ['PCE_pct'] * 100, 'Voc': FINAL['Voc'] / SQ['Voc_V'] * 100,
               'Jsc': FINAL['Jsc'] / SQ['Jsc_mA_cm2'] * 100, 'FF': FINAL['FF'] / SQ['FF_pct'] * 100}

print(f'Spectrum: {P_IN:.1f} W/m2. Maximum of the limit: {PCE_SCAN.max():.2f}% at {EG_SCAN[PCE_SCAN.argmax()]:.3f} eV')
print(f"Limit at Eg = {EG} eV: PCE {SQ['PCE_pct']:.2f}%, Voc {SQ['Voc_V']:.4f} V, "
      f"Jsc {SQ['Jsc_mA_cm2']:.2f} mA/cm2, FF {SQ['FF_pct']:.2f}%")
for k, v in SHARE_OF_SQ.items():
    print(f'  final device {k:<4} = {v:5.1f}% of the limit')
print(f"Voc deficit: {(SQ['Voc_V'] - FINAL['Voc']) * 1000:.0f} mV")

SQ_DIR = os.path.join(OUT_DIR, 'shockley_queisser')
os.makedirs(SQ_DIR, exist_ok=True)
with open(os.path.join(SQ_DIR, 'sq_comparison.json'), 'w') as f:
    json.dump({'spectrum_power_W_m2': P_IN, 'cell_temperature_K': T_CELL,
               'maximum': {'Eg_eV': float(EG_SCAN[PCE_SCAN.argmax()]), 'PCE_pct': float(PCE_SCAN.max())},
               'limit_at_absorber_Eg': SQ, 'final_device_share_of_limit_pct': SHARE_OF_SQ}, f, indent=2)

### 5.9 Figures of the Shockley–Queisser comparison

In [ ]:
use_style({'font.family': 'Times New Roman', 'font.size': 14, 'font.weight': 'bold', 'axes.labelsize': 18,
           'axes.labelweight': 'bold', 'xtick.labelsize': 14, 'ytick.labelsize': 14, 'axes.linewidth': 1.8,
           'xtick.major.width': 1.6, 'ytick.major.width': 1.6, 'xtick.major.size': 7, 'ytick.major.size': 7,
           'legend.fontsize': 13, 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman:bold',
           'mathtext.it': 'Times New Roman:italic:bold', 'mathtext.bf': 'Times New Roman:bold'})

def bold_ticks_and_legend(ax, legend):
    for lab in ax.get_xticklabels() + ax.get_yticklabels() + legend.get_texts():
        lab.set_fontweight('bold')

fig, ax = plt.subplots(figsize=(6.8, 5.4))
ax.plot(EG_SCAN, PCE_SCAN, '-', color='#1f77b4', lw=2.8, label='Shockley–Queisser limit')
ax.axvline(EG, color='red', ls='--', lw=1.5, alpha=0.8)
ax.plot(EG, SQ['PCE_pct'], 'o', color='red', markersize=10, zorder=5, label=f'SQ at E$_g$ = {EG:.2f} eV')
ax.plot(EG, FINAL['PCE'], '^', color='green', markersize=11, zorder=6, markeredgecolor='black',
        markeredgewidth=1.0, label='Optimized device')
ax.set_xlabel('Bandgap E$_g$ (eV)'); ax.set_ylabel('PCE (%)')
ax.set_xlim(0.5, 3.0); ax.grid(True, alpha=0.25)
bold_ticks_and_legend(ax, ax.legend(loc='lower right'))
plt.tight_layout()
save_figure(fig, 'sq_limit_curve')
plt.show()

fig, ax = plt.subplots(figsize=(6.8, 5.4))
labels = ['PCE', r'V$_{\mathrm{OC}}$', r'J$_{\mathrm{SC}}$', 'FF']
x, w = np.arange(4), 0.36
ax.bar(x - w / 2, [100] * 4, w, label='SQ limit', color='#FFD700', edgecolor='black', lw=1.0)
bars = ax.bar(x + w / 2, list(SHARE_OF_SQ.values()), w, label='Optimized device', color='#1f77b4',
              edgecolor='black', lw=1.0)
for b, v in zip(bars, SHARE_OF_SQ.values()):
    ax.text(b.get_x() + b.get_width() / 2, v + 1.2, f'{v:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Percentage of SQ limit (%)'); ax.set_ylim(0, 115)
ax.axhline(100, color='red', ls='--', lw=1.2, alpha=0.6); ax.grid(axis='y', alpha=0.25)
bold_ticks_and_legend(ax, ax.legend(loc='lower left'))
plt.tight_layout()
save_figure(fig, 'device_vs_sq_limit')
plt.show()

### 5.10 Shockley–Queisser limit of common tin-perovskite absorbers
For comparison with absorbers of smaller bandgap.

In [ ]:
print(f"{'Absorber':<22} {'Eg (eV)':>8} {'Jsc':>7} {'Voc':>7} {'PCE':>7}")
for name, Eg in [('MASnI3, CsSnI3', 1.30), ('FASnI3', 1.40), ('MASnIBr2 (this work)', 1.75)]:
    r = sq_limit(Eg)
    print(f"{name:<22} {Eg:8.2f} {r['Jsc_mA_cm2']:7.1f} {r['Voc_V']:7.3f} {r['PCE_pct']:6.1f}%")

### 5.11 PCE on a grid of the three thicknesses
11 × 11 × 11 = 1331 combinations of absorber, WO$_3$ and CBTS thickness; the other variables keep their final values.
Combinations that fail the precheck are not simulated. The ten best combinations are saved as `.scaps` files (about 7 h).

In [ ]:
GRID_DIR = os.path.join(OUT_DIR, 'thickness_grid')
TOP10_DIR = os.path.join(GRID_DIR, 'top10_scaps_files')
os.makedirs(TOP10_DIR, exist_ok=True)
ABS_GRID_NM = list(range(200, 1401, 120))
ETL_GRID_NM = [20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 100]
HTL_GRID_NM = [30, 50, 70, 90, 120, 150, 180, 210, 240, 270, 300]

t0, rows = time.time(), []
for abs_nm in ABS_GRID_NM:
    for etl_nm in ETL_GRID_NM:
        for htl_nm in HTL_GRID_NM:
            row = {'abs_nm': abs_nm, 'etl_nm': etl_nm, 'htl_nm': htl_nm}
            layers, ifaces = device_updates(FINAL, abs_nm / 1000, etl_nm / 1000, htl_nm / 1000)
            check = physics_precheck_detailed(layers, ifaces, device)
            row['precheck_valid'] = check['valid']
            if not check['valid']:
                rows.append({**row, 'reason': check['reason']})
                continue
            path = os.path.join(GRID_DIR, '_grid_point.scaps')
            build_device(path, layers, ifaces)
            r = run_scaps(path)
            table = read_iv_table(OUTPUT_IV_FILE) if r else None
            if table is None:
                rows.append({**row, 'reason': 'SCAPS-1D failed'})
                continue
            Voc, Jsc, FF, eta = r
            rows.append({**row, 'PCE': eta, 'Voc': Voc, 'Jsc': Jsc, 'FF': FF,
                         'j_total_gen': current_at(table, 'j_total_gen(mA/cm2)', 0.0),
                         'j_total_rec_Voc': current_at(table, 'j_total_rec(mA/cm2)', Voc),
                         'jbulk_Voc': current_at(table, 'jbulk(mA/cm2)', Voc),
                         'jifr_Voc': current_at(table, 'jifr(mA/cm2)', Voc)})
    print(f'absorber {abs_nm} nm done ({(time.time() - t0) / 60:.0f} min)')

grid = pd.DataFrame(rows)
grid.to_csv(os.path.join(GRID_DIR, 'thickness_grid.csv'), index=False)
simulated = grid[grid.PCE.notna()]
for rank, (_, r) in enumerate(simulated.nlargest(10, 'PCE').iterrows(), 1):
    a_nm, e_nm, h_nm = int(r.abs_nm), int(r.etl_nm), int(r.htl_nm)
    name = f'rank{rank:02d}_a{a_nm}_e{e_nm}_h{h_nm}_PCE{r.PCE:.4f}.scaps'
    build_device(os.path.join(TOP10_DIR, name), *device_updates(FINAL, a_nm / 1000, e_nm / 1000, h_nm / 1000))
    print(f'rank {rank:2d}: {a_nm}/{e_nm}/{h_nm} nm, PCE {r.PCE:.4f}%')

GRID_SUMMARY = {'n_total': len(grid), 'n_rejected_precheck': int((~grid.precheck_valid).sum()),
                'n_simulated': len(simulated), 'PCE_min': float(simulated.PCE.min()),
                'PCE_max': float(simulated.PCE.max()), 'PCE_mean': float(simulated.PCE.mean()),
                'PCE_sd': float(simulated.PCE.std())}
with open(os.path.join(GRID_DIR, 'thickness_grid_summary.json'), 'w') as f:
    json.dump(GRID_SUMMARY, f, indent=2)
print(GRID_SUMMARY)

### 5.12 PCE maps versus WO$_3$ and CBTS thickness
Three absorber thicknesses of the grid (440, 800 and 1160 nm). Grey cells were rejected by the precheck; the red dot marks
the best cell of each map.

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle

grid = pd.read_csv(os.path.join(GRID_DIR, 'thickness_grid.csv'))

def cell_edges(values):
    v = np.array(sorted(values), float)
    mids = (v[:-1] + v[1:]) / 2
    return np.concatenate([[v[0] - (v[1] - v[0]) / 2], mids, [v[-1] + (v[-1] - v[-2]) / 2]])

use_style({'font.family': 'Times New Roman', 'font.size': 14, 'font.weight': 'bold', 'axes.labelsize': 15,
           'axes.labelweight': 'bold', 'axes.titlesize': 16, 'axes.titleweight': 'bold', 'xtick.labelsize': 13.5,
           'ytick.labelsize': 13.5, 'axes.linewidth': 1.3, 'xtick.major.width': 1.1, 'ytick.major.width': 1.1,
           'xtick.major.size': 5, 'ytick.major.size': 5})

for abs_nm in (440, 800, 1160):
    sub = grid[grid.abs_nm == abs_nm]
    htl, etl = np.array(sorted(sub.htl_nm.unique()), float), np.array(sorted(sub.etl_nm.unique()), float)
    pce = np.full((len(etl), len(htl)), np.nan)
    for _, r in sub[sub.precheck_valid & (sub.PCE > 0)].iterrows():
        pce[np.searchsorted(etl, r.etl_nm), np.searchsorted(htl, r.htl_nm)] = r.PCE
    vmin, vmax = np.nanmin(pce), np.nanmax(pce)
    he, ee = cell_edges(htl), cell_edges(etl)

    fig, ax = plt.subplots(figsize=(5.2, 4.9), constrained_layout=True)
    im = ax.pcolormesh(he, ee, pce, cmap='viridis', shading='flat', vmin=vmin, vmax=vmax)
    for i, j in zip(*np.where(np.isnan(pce))):
        ax.add_patch(Rectangle((he[j], ee[i]), he[j + 1] - he[j], ee[i + 1] - ee[i],
                               facecolor='#d9d9d9', edgecolor='none', zorder=3))
    if vmax - vmin > 0.01:
        H, E = np.meshgrid(htl, etl)
        cs = ax.contour(H, E, pce, levels=np.linspace(vmin, vmax, 6)[1:-1], colors='black', linewidths=1.2, zorder=4)
        for t in ax.clabel(cs, inline=True, fmt='%.2f', fontsize=13, colors='black'):
            t.set_fontweight('bold')
            t.set_path_effects([pe.Stroke(linewidth=3.0, foreground='white', alpha=0.9), pe.Normal()])
    bi = np.unravel_index(np.nanargmax(pce), pce.shape)
    ax.plot(htl[bi[1]], etl[bi[0]], 'o', markersize=11, markerfacecolor='red', markeredgecolor='white',
            markeredgewidth=1.8, zorder=6, label=f'Slice best: {pce[bi]:.2f}%')
    for t in ax.legend(loc='upper left', fontsize=13, framealpha=0.92, edgecolor='0.55').get_texts():
        t.set_fontweight('bold')
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label('PCE (%)', fontsize=15, fontweight='bold')
    cb.ax.tick_params(labelsize=13, width=1.0, length=4)
    for t in cb.ax.get_yticklabels():
        t.set_fontweight('bold')
    ax.set_title(f'Absorber = {abs_nm} nm', pad=10)
    ax.set_xlabel('HTL thickness (nm)'); ax.set_ylabel('ETL thickness (nm)')
    ax.set_xticks([30, 90, 150, 210, 270]); ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_xlim(he[0], he[-1]); ax.set_ylim(ee[0], ee[-1])
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight('bold')
    save_figure(fig, f'pce_map_absorber_{abs_nm}nm')
    plt.show()
    print(f'absorber {abs_nm} nm: PCE {vmin:.3f}–{vmax:.3f}%, best at WO3 {etl[bi[0]]:.0f} nm / '
          f'CBTS {htl[bi[1]]:.0f} nm, {int(np.isnan(pce).sum())} cells rejected')

## Part 6 — Which variables matter most

### 6.1 SHAP and fANOVA importance
Both methods use the PPSR trials. SHAP explains an XGBoost model of the PCE (mean absolute SHAP value per variable);
fANOVA gives the share of the PCE variance that each variable explains. Densities are used as log$_{10}$ values.

In [ ]:
from optuna.importance import FanovaImportanceEvaluator

IMPORTANCE_VARS = ['MASnIBr2_t', 'WO3_t', 'CBTS_t', 'WO3_ND', 'CBTS_NA', 'MASnIBr2_defect',
                   'WO3_defect', 'CBTS_defect', 'iface_CBTS_MASnIBr2_Nt', 'iface_MASnIBr2_WO3_Nt']
LABELS = {'MASnIBr2_t': 'Absorber thickness', 'WO3_t': 'WO$_3$ thickness', 'CBTS_t': 'CBTS thickness',
          'WO3_ND': 'WO$_3$ doping', 'CBTS_NA': 'CBTS doping', 'MASnIBr2_defect': 'Absorber defect density',
          'WO3_defect': 'WO$_3$ defect density', 'CBTS_defect': 'CBTS defect density',
          'iface_CBTS_MASnIBr2_Nt': 'CBTS/MASnIBr$_2$ interface', 'iface_MASnIBr2_WO3_Nt': 'MASnIBr$_2$/WO$_3$ interface'}

X_imp = VALID['PPSR'][IMPORTANCE_VARS].astype(float).copy()
for v in LOG_VARS:
    X_imp[v] = np.log10(X_imp[v].clip(lower=1e-30))
y_imp = VALID['PPSR']['eta'].astype(float)

xgb_model = XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.04, subsample=0.85, colsample_bytree=0.85,
                         random_state=42, objective='reg:squarederror', verbosity=0).fit(X_imp.values, y_imp.values)
background = X_imp.sample(n=min(500, len(X_imp)), random_state=42)
shap_values = shap.Explainer(xgb_model, background.values)(X_imp.values)
shap_imp = pd.Series(np.abs(shap_values.values).mean(axis=0), index=IMPORTANCE_VARS).sort_values(ascending=False)

distributions = {v: optuna.distributions.FloatDistribution(float(X_imp[v].min()), float(X_imp[v].max()))
                 for v in IMPORTANCE_VARS}
study = optuna.create_study(direction='maximize')
for i in range(len(X_imp)):
    study.add_trial(optuna.trial.create_trial(params={v: float(X_imp.iloc[i][v]) for v in IMPORTANCE_VARS},
                                              distributions=distributions, value=float(y_imp.iloc[i])))
fanova_imp = pd.Series(FanovaImportanceEvaluator(seed=42).evaluate(study)).sort_values(ascending=False)

IMP_DIR = os.path.join(OUT_DIR, 'variable_importance')
os.makedirs(IMP_DIR, exist_ok=True)
shap_imp.rename('mean_abs_shap').to_csv(os.path.join(IMP_DIR, 'shap_importance.csv'))
fanova_imp.rename('fanova_importance').to_csv(os.path.join(IMP_DIR, 'fanova_importance.csv'))
print(pd.DataFrame({'SHAP': shap_imp, 'fANOVA': fanova_imp}).sort_values('SHAP', ascending=False).round(4))

use_style({'font.size': 10, 'axes.labelsize': 11, 'axes.labelweight': 'bold', 'axes.linewidth': 1.3,
           'xtick.major.size': 5, 'ytick.major.size': 5, 'xtick.major.width': 1.1, 'ytick.major.width': 1.1})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 4.5), gridspec_kw={'wspace': 0.72})
for ax, imp, color, xlabel in [(ax1, shap_imp, '#1f77b4', 'Mean absolute SHAP value'),
                               (ax2, fanova_imp, '#ff7f0e', 'fANOVA variance contribution')]:
    s = imp.sort_values()
    ax.barh([LABELS[v] for v in s.index], s.values, color=color, edgecolor='black', lw=0.6, alpha=0.90)
    ax.set_xlabel(xlabel, fontsize=11, fontweight='bold')
    ax.set_xlim(0, s.max() * 1.18)
    for i, v in enumerate(s.values):
        ax.text(v + s.max() * 0.018, i, '<0.001' if 0 < v < 0.001 else f'{v:.3f}',
                va='center', ha='left', fontsize=8.5, fontweight='bold')
    ax.grid(axis='x', ls='--', lw=0.5, alpha=0.35); ax.set_axisbelow(True)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight('bold')
plt.subplots_adjust(left=0.20, right=0.98, top=0.95, bottom=0.25, wspace=0.72)
for ax, title in [(ax1, '(a) SHAP-based local feature importance'), (ax2, '(b) fANOVA-based global feature importance')]:
    pos = ax.get_position()
    fig.text(pos.x0 + pos.width / 2, 0.095, title, ha='center', va='center', fontsize=11, fontweight='bold')
save_figure(fig, 'variable_importance_shap_fanova')
plt.show()
del xgb_model, shap_values, study

### 6.2 Correlation between the variables and the results
Pearson, Spearman and Kendall correlation coefficients of the ten variables and Voc, Jsc, FF and PCE (PPSR trials).

In [ ]:
CORR_NAMES = {'MASnIBr2_t': 'Absorber t', 'WO3_t': 'ETL t', 'CBTS_t': 'HTL t', 'WO3_ND': r'ETL $N_D$',
              'CBTS_NA': r'HTL $N_A$', 'MASnIBr2_defect': r'Absorber $N_t$', 'WO3_defect': r'ETL $N_t$',
              'CBTS_defect': r'HTL $N_t$', 'iface_CBTS_MASnIBr2_Nt': 'HTL/Abs IFD', 'iface_MASnIBr2_WO3_Nt': 'Abs/ETL IFD',
              'Voc': r'$V_{oc}$', 'Jsc': r'$J_{sc}$', 'FF': 'FF', 'eta': 'PCE'}
corr_data = VALID['PPSR'][list(CORR_NAMES)].replace([np.inf, -np.inf], np.nan).dropna().rename(columns=CORR_NAMES)

use_style({'font.family': 'Times New Roman', 'font.size': 17, 'font.weight': 'bold', 'axes.labelweight': 'bold',
           'axes.titleweight': 'bold', 'axes.linewidth': 1.3})
for method in ('pearson', 'spearman', 'kendall'):
    corr = corr_data.corr(method=method)
    corr.to_csv(os.path.join(IMP_DIR, f'{method}_correlation_matrix.csv'))
    n = len(corr)
    fig, ax = plt.subplots(figsize=(11.0, 9.4))
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
    ax.set_xticks(np.arange(n)); ax.set_yticks(np.arange(n))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right', rotation_mode='anchor'); ax.set_yticklabels(corr.columns)
    ax.set_xticks(np.arange(-0.5, n, 1), minor=True); ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
    ax.grid(which='minor', color='white', ls='-', lw=0.8); ax.tick_params(which='minor', bottom=False, left=False)
    for i in range(n):
        for j in range(n):
            val = corr.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=14, fontweight='bold',
                    color='white' if abs(val) >= 0.55 else 'black')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.045)
    cbar.set_label('Correlation coefficient', fontweight='bold', rotation=90, labelpad=14)
    cbar.outline.set_linewidth(1.2)
    for lab in ax.get_xticklabels() + ax.get_yticklabels() + cbar.ax.get_yticklabels():
        lab.set_fontweight('bold')
    ax.set_title(f'{method.capitalize()} correlation heatmap', fontsize=20, fontweight='bold', pad=18)
    for spine in ax.spines.values():
        spine.set_linewidth(1.4)
    plt.tight_layout()
    save_figure(fig, f'{method}_correlation_heatmap')
    plt.show()

### 6.3 Sobol indices
First-order Sobol indices tell how much of the PCE variance each variable explains on its own when all variables are
sampled evenly within their bounds. They are computed on an Extra Trees model of all simulated trials
(Saltelli estimator, N = 80 000, three repeats) for three ranges of the absorber defect density:
the full range, 10$^{13}$–10$^{14}$ cm$^{-3}$, and the narrow range that PPSR reached (10$^{13}$–1.2 × 10$^{13}$ cm$^{-3}$).
The upper bound of the CBTS doping is the CBTS effective density of states, 1.8 × 10$^{19}$ cm$^{-3}$ (rule C2).

In [ ]:
SOBOL_BOUNDS = {'WO3_t': (0.02, 0.1), 'MASnIBr2_t': (0.3, 1.4), 'CBTS_t': (0.03, 0.3),
                'WO3_ND': (1e16, 1e19), 'CBTS_NA': (1e17, 1.8e19), 'MASnIBr2_defect': (1e13, 1e16),
                'WO3_defect': (1e13, 1e16), 'CBTS_defect': (1e13, 1e16),
                'iface_CBTS_MASnIBr2_Nt': (1e9, 1e12), 'iface_MASnIBr2_WO3_Nt': (1e9, 1e12)}

def log_X(df):
    X = df[VARS].astype(float).copy()
    for v in LOG_VARS:
        X[v] = np.log10(X[v])
    return X.values

sobol_model = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, random_state=0, n_jobs=-1).fit(
    log_X(all_valid), all_valid['eta'].values)

def sample_uniform(N, bounds, rng):
    S = np.empty((N, len(VARS)))
    for j, v in enumerate(VARS):
        lo, hi = bounds[v]
        S[:, j] = rng.uniform(np.log10(lo), np.log10(hi), N) if v in LOG_VARS else rng.uniform(lo, hi, N)
    return S

def sobol_first_order(bounds, rng, N=80000):
    A, B = sample_uniform(N, bounds, rng), sample_uniform(N, bounds, rng)
    fA, fB = sobol_model.predict(A), sobol_model.predict(B)
    V = np.var(np.r_[fA, fB])
    S1 = {}
    for j, v in enumerate(VARS):
        AB = A.copy(); AB[:, j] = B[:, j]
        S1[v] = np.mean(fB * (sobol_model.predict(AB) - fA)) / V
    return S1

cases = {'full bounds': SOBOL_BOUNDS,
         'absorber Nt 1e13–1e14 cm-3': {**SOBOL_BOUNDS, 'MASnIBr2_defect': (1e13, 1e14)},
         'absorber Nt 1e13–1.2e13 cm-3 (PPSR range)': {**SOBOL_BOUNDS, 'MASnIBr2_defect': (1e13, 1.2e13)}}
repeats = {name: [] for name in cases}
for r in range(3):
    rng = np.random.default_rng(100 + r)
    for name, bounds in cases.items():
        repeats[name].append(sobol_first_order(bounds, rng))

for name, reps in repeats.items():
    S1 = {v: np.mean([s[v] for s in reps]) for v in VARS}
    print(f'\n{name}: four largest first-order indices (mean and range of 3 repeats)')
    for v in sorted(VARS, key=lambda v: -S1[v])[:4]:
        print(f'  {v:<24} S1 = {S1[v]:.3f}  [{min(s[v] for s in reps):.3f}, {max(s[v] for s in reps):.3f}]')
del sobol_model

## Part 7 — Robustness: Monte Carlo and material parameters

### 7.1 Monte Carlo fabrication tolerance (about 2 h)
500 random devices are drawn around the final device. Each of the ten variables is drawn from a log-normal distribution
with the relative spread given in `MC_INPUT_SIGMA` (step 0.3). Devices that fail the precheck are not simulated.
First, the unchanged final device is simulated as a check: it must give the PCE of step 5.1.

In [ ]:
FINAL_DB = os.path.join(OUT_DIR, {'PPSR': 'trials.db', 'TPE': 'baseline_trials.db', 'GA': 'ga_trials.db'}[FINAL['algorithm']])
nominal = _get_mc_nominal_params(FINAL_DB)
check_file = os.path.join(OUT_DIR, 'mc_nominal_check.scaps')
layers, ifaces = _mc_build_updates(nominal)
build_updated_scaps_file(TEMPLATE_FILE, check_file, layers, ifaces, back_contact_wf=nominal['back_contact_wf'], verify=False)
pce_check = run_scaps(check_file)[3]
print(f"Unchanged final device: PCE = {pce_check:.4f}% (step 5.1: {FINAL['PCE']:.4f}%)")
assert abs(pce_check - FINAL['PCE']) < 0.01

mc_result = run_mc_uncertainty(FINAL_SCAPS, nominal_db=FINAL_DB, summary_name='mc_summary.json')
MC = mc_result['samples']
MC_STATS = mc_result['stats']

### 7.2 Distribution of the Monte Carlo PCE

In [ ]:
eta = MC['eta'].dropna().values
mean_pce, sd_pce = eta.mean(), eta.std(ddof=1)
p05, p50, p95 = np.percentile(eta, [5, 50, 95])
print(f'N = {len(eta)} of {MC_N_SAMPLES} draws simulated; PCE {mean_pce:.3f} ± {sd_pce:.3f}%, '
      f'5th / 50th / 95th percentile {p05:.3f} / {p50:.3f} / {p95:.3f}%')

use_style({'font.size': 11, 'axes.labelsize': 12, 'axes.labelweight': 'bold', 'axes.linewidth': 1.2,
           'xtick.labelsize': 10, 'ytick.labelsize': 10, 'xtick.major.size': 5, 'ytick.major.size': 5,
           'xtick.major.width': 1.1, 'ytick.major.width': 1.1})
fig, ax = plt.subplots(figsize=(6.6, 4.5))
counts, _, _ = ax.hist(eta, bins=32, color='#4C93B6', alpha=0.78, edgecolor='black', lw=0.55, label='Monte Carlo samples')
ax.axvspan(p05, p95, color='#2ca02c', alpha=0.13, label='90% uncertainty interval')
ax.axvline(mean_pce, color='red', ls='-', lw=1.8, label=f'Mean = {mean_pce:.3f}%')
ax.axvline(p50, color='black', ls=':', lw=1.5, label=f'Median = {p50:.3f}%')
ax.axvline(p05, color='darkgreen', ls='--', lw=1.4, label=f'5th percentile = {p05:.3f}%')
ax.axvline(p95, color='darkgreen', ls='--', lw=1.4, label=f'95th percentile = {p95:.3f}%')
ax.set_xlabel('Power conversion efficiency, PCE (%)'); ax.set_ylabel('Frequency')
pad = 0.04 * (eta.max() - eta.min())
ax.set_xlim(eta.min() - pad, eta.max() + pad); ax.set_ylim(0, counts.max() * 1.16)
ax.grid(axis='y', ls='--', lw=0.5, alpha=0.35); ax.set_axisbelow(True)
legend = ax.legend(loc='upper left', fontsize=8.6, borderpad=0.55, handlelength=2.3)
for lab in ax.get_xticklabels() + ax.get_yticklabels() + legend.get_texts():
    lab.set_fontweight('bold')
ax.text(0.98, 0.64, f'N = {len(eta)} (of {MC_N_SAMPLES} draws)\nMean ± SD = {mean_pce:.3f} ± {sd_pce:.3f}%\n'
        f'90% interval = [{p05:.3f}, {p95:.3f}]%', transform=ax.transAxes, ha='right', va='top', fontsize=9,
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='gray', alpha=0.88))
plt.subplots_adjust(left=0.12, right=0.98, top=0.96, bottom=0.14)
save_figure(fig, 'monte_carlo_pce')
plt.show()

### 7.3 Uncertain material parameters (about 15 min)
The material parameters taken from the literature are changed one at a time in the `.scaps` file of the final device
(mobility, electron affinities, dielectric constant, densities of states, capture cross-sections, defect density and level).
Then the best absorber thickness is found for four absorber defect densities.

The functions below read and change single values in a `.scaps` file; `run_text()` simulates a changed file.

In [ ]:
def read_text(path):
    raw = open(path, 'rb').read()
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            return raw.decode(enc), enc
        except UnicodeDecodeError:
            pass
    raise ValueError(path)

def _block_span(text, header_regex, end_regex):
    m = re.search(header_regex, text, flags=re.M)
    if not m:
        raise KeyError(header_regex)
    e = re.search(end_regex, text[m.end():], flags=re.M)
    return m.start(), (m.end() + e.start()) if e else len(text)

def layer_span(text, layer):
    return _block_span(text, r'^name\s*:\s*' + re.escape(layer) + r'\s*$', r'^(layer|front contact)\s*$')

def iface_span(text, iface):
    return _block_span(text, r'^interfacename\s*:\s*' + re.escape(iface) + r'\s*$', r'^layer\s*$')

def get_table_param(text, layer, key):
    s, e = layer_span(text, layer)
    return float(re.search(r'^' + re.escape(key) + r'\s*:\t\s*([-+0-9.eE]+)', text[s:e], re.M).group(1))

def set_table_param(text, layer, key, value_SI):
    """Replace a tabulated layer property (e.g. 'mu_n', 'chi', 'eps', 'Nc'): the pure-material value and its two copies."""
    s, e = layer_span(text, layer)
    blk = text[s:e]
    m = re.compile(r'^(' + re.escape(key) + r'\s*:\t)(.*)$', re.M).search(blk)
    if not m:
        raise KeyError(f'{layer}:{key}')
    fields = m.group(2).split('\t')
    num_idx = [i for i, f in enumerate(fields) if re.fullmatch(r'\s*[-+0-9.eE]+\s*', f)]
    fmt = (lambda v: f' {v:.6e}') if abs(value_SI) >= 1e3 or abs(value_SI) < 1e-2 else (lambda v: f'  {v:.6f}')
    for pos in (0, 5, 6):
        fields[num_idx[pos]] = fmt(value_SI)
    blk = blk[:m.start(2)] + '\t'.join(fields) + blk[m.end(2):]
    return text[:s] + blk + text[e:]

def get_single(text, span, key):
    s, e = span
    return float(re.search(r'^' + re.escape(key) + r'\s*:\s*([-+0-9.eE]+)', text[s:e], re.M).group(1))

def set_single(text, span, key, value, fmt='{:.3e}'):
    s, e = span
    blk, n = re.subn(r'^(' + re.escape(key) + r'\s*:\s*)[-+0-9.eE]+', lambda m: m.group(1) + fmt.format(value),
                     text[s:e], count=1, flags=re.M)
    if n != 1:
        raise KeyError(key)
    return text[:s] + blk + text[e:]

base_text, ENCODING = read_text(FINAL_SCAPS)

def run_text(text, path):
    with open(path, 'w', encoding=ENCODING, newline='') as f:
        f.write(text)
    return run_scaps(path)

The one-at-a-time changes and the "tornado" summary (largest PCE range first):

In [ ]:
SENS_DIR = os.path.join(OUT_DIR, 'material_sensitivity')
os.makedirs(SENS_DIR, exist_ok=True)
A, E, H = 'MASnIBr2', 'WO3', 'CBTS'

cases = []
mu0 = get_table_param(base_text, A, 'mu_n')
for f in (0.25, 0.5, 2.0, 4.0):
    cases.append(('Absorber mobility (mu_n = mu_p)', f'x{f:g}', [('table', A, 'mu_n', mu0 * f), ('table', A, 'mu_p', mu0 * f)]))
for d in (-0.2, -0.1, 0.1, 0.2):
    cases.append(('Absorber electron affinity', f'{d:+.1f} eV', [('table', A, 'chi', get_table_param(base_text, A, 'chi') + d)]))
    cases.append(('WO3 electron affinity (CBO)', f'{d:+.1f} eV', [('table', E, 'chi', get_table_param(base_text, E, 'chi') + d)]))
    cases.append(('CBTS electron affinity (VBO)', f'{d:+.1f} eV', [('table', H, 'chi', get_table_param(base_text, H, 'chi') + d)]))
for v in (6.0, 10.0, 12.0):
    cases.append(('Absorber dielectric constant', f'eps={v:g}', [('table', A, 'eps', v)]))
nc0, nv0 = get_table_param(base_text, A, 'Nc'), get_table_param(base_text, A, 'Nv')
for f in (0.1, 10.0):
    cases.append(('Absorber Nc = Nv', f'x{f:g}', [('table', A, 'Nc', nc0 * f), ('table', A, 'Nv', nv0 * f)]))
sigma0 = get_single(base_text, layer_span(base_text, A), 'sigma_n')
for f in (10.0, 100.0):
    cases.append(('Absorber capture cross-section', f'x{f:g}',
                  [('layer', A, 'sigma_n', sigma0 * f), ('layer', A, 'sigma_p', sigma0 * f)]))
for nt in (3e13, 1e14, 1e15):
    cases.append(('Absorber defect density Nt', f'{nt:.0e} cm-3', [('table', A, 'Nt(uniform)', nt * 1e6)]))
et0 = get_single(base_text, layer_span(base_text, A), 'Et')
for d in (-0.3, 0.3):
    cases.append(('Absorber defect level Et', f'{d:+.1f} eV', [('layer', A, 'Et', et0 + d)]))
for iface in ('CBTS / MASnIBr2', 'MASnIBr2 / WO3'):
    sp = iface_span(base_text, iface)
    sig = {k: get_single(base_text, sp, k) for k in ('sigma_nleft', 'sigma_nright', 'sigma_pleft', 'sigma_pright')}
    for f in (10.0, 100.0):
        cases.append((f'Interface sigma ({iface})', f'x{f:g}', [('iface', iface, k, v * f) for k, v in sig.items()]))

def apply_case(text, changes):
    for kind, where, key, value in changes:
        if kind == 'table':
            text = set_table_param(text, where, key, value)
        elif kind == 'layer':
            text = set_single(text, layer_span(text, where), key, value, '{:.3f}' if key == 'Et' else '{:.3e}')
        else:
            text = set_single(text, iface_span(text, where), key, value)
    return text

ref = run_text(base_text, os.path.join(SENS_DIR, 'case_000_reference.scaps'))
print(f'Reference (final device): PCE = {ref[3]:.4f}%')
rows = []
for i, (group, level, changes) in enumerate(cases, 1):
    r = run_text(apply_case(base_text, changes), os.path.join(SENS_DIR, f'case_{i:03d}.scaps'))
    if r is None:
        print(f'  {group:<36} {level:<12} SCAPS-1D failed')
        continue
    rows.append(dict(group=group, level=level, Voc=r[0], Jsc=r[1], FF=r[2], PCE=r[3], dPCE=r[3] - ref[3]))
    print(f'  {group:<36} {level:<12} PCE = {r[3]:.3f}%  change {r[3] - ref[3]:+.3f} pp')

sens = pd.DataFrame(rows)
sens.to_csv(os.path.join(SENS_DIR, 'material_sensitivity.csv'), index=False)
tornado = (sens.groupby('group').dPCE.agg(['min', 'max']).assign(span=lambda d: d['max'] - d['min'])
           .sort_values('span', ascending=False))
tornado.to_csv(os.path.join(SENS_DIR, 'material_sensitivity_summary.csv'))
print(tornado.round(3))

use_style()
fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(tornado) + 1.2))
y = np.arange(len(tornado))[::-1]
ax.barh(y, tornado['max'], color='#2a6fdb', height=0.6, label='increase')
ax.barh(y, tornado['min'], color='#d1495b', height=0.6, label='decrease')
ax.axvline(0, color='k', lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(tornado.index, fontsize=9)
ax.set_xlabel('Change in PCE from the final device (percentage points)')
ax.set_title(f'Material-parameter sensitivity (reference PCE {ref[3]:.2f}%)', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save_figure(fig, 'material_sensitivity_tornado')
plt.show()

Best absorber thickness for four absorber defect densities:

In [ ]:
grid_nt = []
for nt in (1e13, 1e14, 1e15, 1e16):
    for d_nm in (300, 500, 700, 900, 1100, 1300):
        text = set_table_param(base_text, A, 'Nt(uniform)', nt * 1e6)
        text = set_single(text, layer_span(text, A), 'd', d_nm * 1e-9, '{:.3e}')
        r = run_text(text, os.path.join(SENS_DIR, f'grid_Nt{nt:.0e}_d{d_nm}.scaps'))
        if r is not None:
            grid_nt.append(dict(Nt_cm3=nt, d_nm=d_nm, Voc=r[0], Jsc=r[1], FF=r[2], PCE=r[3]))
grid_nt = pd.DataFrame(grid_nt)
grid_nt.to_csv(os.path.join(SENS_DIR, 'thickness_vs_defect_density.csv'), index=False)
print(grid_nt.loc[grid_nt.groupby('Nt_cm3').PCE.idxmax(), ['Nt_cm3', 'd_nm', 'PCE']].to_string(index=False))

use_style()
fig, ax = plt.subplots(figsize=(6, 4))
for nt, g in grid_nt.groupby('Nt_cm3'):
    ax.plot(g.d_nm, g.PCE, 'o-', label=f'N$_t$ = {nt:.0e} cm$^{{-3}}$')
ax.set_xlabel('MASnIBr$_2$ thickness (nm)'); ax.set_ylabel('PCE (%)'); ax.legend(fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save_figure(fig, 'absorber_thickness_vs_defect_density')
plt.show()

### 7.4 WO$_3$ bandgap and FTO density of states
The final device is simulated with four WO$_3$ bandgaps (2.0–3.0 eV), and once with a lower effective density of states
of the FTO conduction band (2.2 × 10$^{18}$ cm$^{-3}$).

In [ ]:
WO3_DIR = os.path.join(OUT_DIR, 'wo3_bandgap_check')
os.makedirs(WO3_DIR, exist_ok=True)
print(f"WO3 in the device file: Eg = {get_table_param(base_text, E, 'Eg')} eV, chi = {get_table_param(base_text, E, 'chi')} eV")

rows = []
for eg in (2.0, 2.6, 2.8, 3.0):
    r = run_text(set_table_param(base_text, E, 'Eg', eg), os.path.join(WO3_DIR, f'WO3_Eg_{eg:.1f}.scaps'))
    rows.append({'case': f'WO3 Eg = {eg:.1f} eV', 'Voc_V': r[0], 'Jsc_mA_cm2': r[1], 'FF_pct': r[2], 'PCE_pct': r[3]})
r = run_text(set_table_param(base_text, 'FTO', 'Nc', 2.2e24), os.path.join(WO3_DIR, 'FTO_Nc_2.2e18.scaps'))
rows.append({'case': 'FTO Nc = 2.2e18 cm-3 (WO3 Eg = 2.0 eV)', 'Voc_V': r[0], 'Jsc_mA_cm2': r[1], 'FF_pct': r[2], 'PCE_pct': r[3]})

wo3 = pd.DataFrame(rows)
wo3['dPCE_pp'] = wo3['PCE_pct'] - wo3['PCE_pct'].iloc[0]
wo3.to_csv(os.path.join(WO3_DIR, 'wo3_bandgap_fto_nc.csv'), index=False)
print(wo3.round(4).to_string(index=False))

## Part 8 — Machine-learning models and their validation

### 8.1 Data set
All successfully simulated trials of the three algorithms. Inputs: the ten design variables (densities as log$_{10}$ values);
output: the PCE (and Voc, Jsc and FF for the stacked model). The same random 80/20 split into training and test data
is used in all following steps.

In [ ]:
ML_DIR = os.path.join(OUT_DIR, 'ml_surrogate')
os.makedirs(ML_DIR, exist_ok=True)
RS = 42

data = all_valid.dropna(subset=VARS + ['eta', 'Voc', 'Jsc', 'FF']).reset_index(drop=True)
X = log_X(data)
y = data['eta'].values
Y = data[['eta', 'Voc', 'Jsc', 'FF']].values
seeds = data['seed'].values

X_train, X_test, y_train, y_test, Y_train, Y_test = train_test_split(X, y, Y, test_size=0.2, random_state=RS)
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=RS)

dataset = data[['algorithm', 'seed', 'trial_number'] + VARS + ['Voc', 'Jsc', 'FF', 'eta']].rename(columns={'eta': 'PCE'})
dataset['split'] = 'train'
dataset.loc[idx_test, 'split'] = 'test'
dataset.to_csv(os.path.join(ML_DIR, 'ml_dataset.csv'), index=False)

def extra_trees(n=500):
    return ExtraTreesRegressor(n_estimators=n, random_state=RS, n_jobs=-1)

def metrics(y_true, y_pred):
    return {'R2': r2_score(y_true, y_pred), 'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))}

def show(label, m):
    print(f"  {label:<40} R² = {m['R2']:.4f}  MAE = {m['MAE']:.4f}  RMSE = {m['RMSE']:.4f}")

print(f'{len(data):,} trials: ' + ', '.join(f'{a} {(data.algorithm == a).sum():,}' for a in ALGOS))
print(f'Training {len(y_train):,}, test {len(y_test):,}')

### 8.2 Eight regression models
Each model is trained on the training data and tested on the test data (PCE error in percentage points).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.base import clone

MODEL_NAMES = ['Linear', 'Ridge', 'Lasso', 'SVR-RBF', 'Random Forest', 'Extra Trees', 'Gradient Boosting', 'Hist Grad Boost']

def ml_model(name):
    """A new, untrained model of the given name."""
    return {
        'Linear':            lambda: Pipeline([('scaler', StandardScaler()), ('m', LinearRegression())]),
        'Ridge':             lambda: Pipeline([('scaler', StandardScaler()), ('m', Ridge(alpha=1.0))]),
        'Lasso':             lambda: Pipeline([('scaler', StandardScaler()), ('m', Lasso(alpha=0.001, max_iter=20000))]),
        'SVR-RBF':           lambda: Pipeline([('scaler', StandardScaler()), ('m', SVR(C=100, gamma='scale', epsilon=0.01))]),
        'Random Forest':     lambda: RandomForestRegressor(n_estimators=500, random_state=RS, n_jobs=-1),
        'Extra Trees':       lambda: extra_trees(),
        'Gradient Boosting': lambda: GradientBoostingRegressor(random_state=RS),
        'Hist Grad Boost':   lambda: HistGradientBoostingRegressor(random_state=RS),
    }[name]()

rows, best_r2 = [], -np.inf
for name in MODEL_NAMES:
    t0 = time.time()
    model = ml_model(name).fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({'model': name, 'train_r2': r2_score(y_train, model.predict(X_train)), 'test_r2': r2_score(y_test, pred),
                 'test_mae': mean_absolute_error(y_test, pred), 'test_rmse': np.sqrt(mean_squared_error(y_test, pred)),
                 'fit_time_s': time.time() - t0})
    if rows[-1]['test_r2'] > best_r2:
        best_r2, BEST_NAME, BEST_TEST_PRED = rows[-1]['test_r2'], name, pred
    del model
MODEL_TABLE = pd.DataFrame(rows)
MODEL_TABLE.to_csv(os.path.join(ML_DIR, 'model_comparison.csv'), index=False)
print(MODEL_TABLE.round(4).to_string(index=False))
print(f'\nBest model: {BEST_NAME}')

### 8.3 Physics-structured stacked model
Two base models: one predicts the PCE directly, the other predicts Voc, Jsc and FF (one Extra Trees model each),
from which PCE = Voc × Jsc × FF / 100 (at 100 mW/cm$^2$). A ridge regression combines the two PCE values; it is fitted on
a quarter of the training data that the base models did not see. The models are trained one after the other and deleted
after use, so that only one large model is in memory at a time.

In [ ]:
def physics_pce(Xfit, Yfit, Xpred, n_trees=500):
    """PCE = Voc x Jsc x FF / 100 from three Extra Trees models (columns 1-3 of Yfit: Voc, Jsc, FF)."""
    p = np.ones(len(Xpred))
    for k in (1, 2, 3):
        p *= extra_trees(n_trees).fit(Xfit, Yfit[:, k]).predict(Xpred)
    return p / 100

def stacked_fit_predict(direct, Xtr, ytr, Ytr, Xte, n_trees=500):
    """Direct PCE, physics PCE and stacked PCE for Xte; 'direct' is an untrained model."""
    Xb, Xv, yb, yv, Yb, Yv = train_test_split(Xtr, ytr, Ytr, test_size=0.25, random_state=RS)
    stack = Ridge(alpha=0.01).fit(np.c_[clone(direct).fit(Xb, yb).predict(Xv), physics_pce(Xb, Yb, Xv, n_trees)], yv)
    p_direct = clone(direct).fit(Xtr, ytr).predict(Xte)
    p_physics = physics_pce(Xtr, Ytr, Xte, n_trees)
    return p_direct, p_physics, stack.predict(np.c_[p_direct, p_physics])

pce_direct, pce_physics, pce_stacked = stacked_fit_predict(ml_model(BEST_NAME), X_train, y_train, Y_train, X_test)

STACKING = pd.DataFrame([{'approach': name, **metrics(y_test, pred)} for name, pred in
                         [('Direct prediction', pce_direct), ('Physics reconstruction', pce_physics),
                          ('Hybrid stacking', pce_stacked)]])
STACKING.to_csv(os.path.join(ML_DIR, 'hybrid_stacking_results.csv'), index=False)
print(STACKING.round(4).to_string(index=False))

### 8.4 Conformal prediction intervals
Split conformal prediction: the best model is trained on three quarters of the training data, and the 90% quantile of
its absolute errors on the remaining quarter gives an interval ±q̂ that should contain 90% of the test values.

In [ ]:
X_fit, X_cal, y_fit, y_cal = train_test_split(X_train, y_train, test_size=0.25, random_state=RS)
conformal_model = ml_model(BEST_NAME).fit(X_fit, y_fit)
q_hat = np.quantile(np.abs(y_cal - conformal_model.predict(X_cal)), 0.90)
cp_pred = conformal_model.predict(X_test)
cp_inside = np.abs(y_test - cp_pred) <= q_hat
del conformal_model

pd.DataFrame({'actual_pce': y_test, 'predicted_pce': cp_pred, 'lower_90': cp_pred - q_hat, 'upper_90': cp_pred + q_hat,
              'in_interval': cp_inside}).to_csv(os.path.join(ML_DIR, 'conformal_predictions.csv'), index=False)
with open(os.path.join(ML_DIR, 'conformal_summary.json'), 'w') as f:
    json.dump({'target_coverage_pct': 90.0, 'empirical_coverage_pct': float(cp_inside.mean() * 100),
               'q_hat': float(q_hat), 'n_calibration': len(y_cal), 'n_test': len(y_test)}, f, indent=2)
print(f'Calibration {len(y_cal):,}, test {len(y_test):,}: interval ±{q_hat:.4f} percentage points, '
      f'coverage {cp_inside.mean() * 100:.2f}% (target 90%)')

### 8.5 Figure: model comparison and predicted versus simulated PCE

In [ ]:
use_style()
table = MODEL_TABLE.sort_values('test_r2')
best_row = MODEL_TABLE.loc[MODEL_TABLE.test_r2.idxmax()]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.8, 4.3), gridspec_kw={'width_ratios': [1.08, 1.0], 'wspace': 0.42})
bars = ax1.barh(table['model'], table['test_r2'], edgecolor='black', lw=0.6,
                color=['#ff7f0e' if m == BEST_NAME else '#1f77b4' for m in table['model']])
for bar, v in zip(bars, table['test_r2']):
    ax1.text(v - 0.012, bar.get_y() + bar.get_height() / 2, f'{v:.4f}', va='center', ha='right', fontsize=8, fontweight='bold')
ax1.set_xlabel('Test R²', fontsize=10, fontweight='bold'); ax1.set_xlim(0.50, 1.02)

ax2.scatter(y_test, BEST_TEST_PRED, s=7, alpha=0.32, color='#1f77b4', edgecolors='none')
ax2.plot([17, 26], [17, 26], '--', color='gray', lw=1.2)
ax2.set_xlim(17, 26); ax2.set_ylim(17, 26); ax2.set_aspect('equal', adjustable='box')
ax2.set_xlabel('SCAPS-computed PCE (%)', fontsize=10, fontweight='bold')
ax2.set_ylabel('ML-predicted PCE (%)', fontsize=10, fontweight='bold')
for ax in (ax1, ax2):
    ax.tick_params(axis='both', labelsize=9)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight('bold')
plt.subplots_adjust(left=0.10, right=0.98, top=0.95, bottom=0.24, wspace=0.42)
for ax, title in [(ax1, '(a) ML model comparison (test R²)'),
                  (ax2, f"(b) Best model: {BEST_NAME}; R² = {best_row.test_r2:.4f}, MAE = {best_row.test_mae:.4f} pp")]:
    pos = ax.get_position()
    fig.text(pos.x0 + pos.width / 2, 0.10, title, ha='center', va='center', fontsize=10, fontweight='bold')
save_figure(fig, 'ml_model_comparison')
plt.show()

### 8.6 Figure: stacked model and conformal intervals
(b) shows the 120 test devices with the lowest predicted PCE.

In [ ]:
use_style({'font.size': 10, 'axes.labelweight': 'bold', 'axes.linewidth': 1.1, 'xtick.major.size': 5,
           'ytick.major.size': 5, 'xtick.major.width': 1.0, 'ytick.major.width': 1.0, 'mathtext.fontset': 'custom',
           'mathtext.rm': 'Times New Roman:bold', 'mathtext.it': 'Times New Roman:italic:bold',
           'mathtext.bf': 'Times New Roman:bold'})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.8, 4.3), gridspec_kw={'wspace': 0.42})
bars = ax1.bar(np.arange(3), STACKING['R2'], color=['#1f77b4', '#2ca02c', '#ff7f0e'], edgecolor='black', lw=0.7, width=0.72)
for bar, r2, mae in zip(bars, STACKING['R2'], STACKING['MAE']):
    ax1.text(bar.get_x() + bar.get_width() / 2, r2 + 0.00045, f'R² = {r2:.4f}\nMAE = {mae:.3f}',
             ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1.set_ylim(0.985, STACKING['R2'].max() + 0.004)
ax1.set_xticks(np.arange(3)); ax1.set_xticklabels(['Direct\nprediction', 'Physics\nreconstruction', 'Hybrid\nstacking'], fontsize=9)
ax1.set_ylabel('Test R²', fontsize=11, fontweight='bold')
ax1.grid(axis='y', ls='--', lw=0.5, alpha=0.35); ax1.set_axisbelow(True)

order = np.argsort(cp_pred)[:min(120, len(y_test))]
xs = np.arange(len(order))
ax2.fill_between(xs, cp_pred[order] - q_hat, cp_pred[order] + q_hat, alpha=0.25, color='#1f77b4',
                 label=f'90% interval, $\\hat{{q}}$ = {q_hat:.3f}')
ax2.plot(xs, cp_pred[order], '-', color='#1f77b4', lw=1.7, label='Predicted')
ax2.scatter(xs, y_test[order], s=14, color='#d62728', alpha=0.70, label='Actual')
ax2.set_xlabel('Sorted test sample index', fontsize=10, fontweight='bold'); ax2.set_ylabel('PCE (%)', fontsize=11, fontweight='bold')
ax2.set_ylim(min((cp_pred[order] - q_hat).min(), y_test[order].min()) - 0.25,
             max((cp_pred[order] + q_hat).max(), y_test[order].max()) + 0.25)
ax2.set_xlim(-2, len(order) + 2)
legend = ax2.legend(loc='lower right', fontsize=7.5, borderpad=0.5, handlelength=2.2)
for ax in (ax1, ax2):
    ax.tick_params(axis='both', labelsize=9)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight('bold')
for t in legend.get_texts():
    t.set_fontweight('bold')
plt.subplots_adjust(left=0.085, right=0.985, top=0.965, bottom=0.25, wspace=0.42)
for ax, title in [(ax1, '(a) Physics-guided hybrid stacking'),
                  (ax2, f'(b) Conformal prediction intervals; empirical coverage = {cp_inside.mean() * 100:.1f}% '
                        'at 90% nominal coverage')]:
    pos = ax.get_position()
    fig.text(pos.x0 + pos.width / 2, 0.105, title, ha='center', va='center', fontsize=10, fontweight='bold')
save_figure(fig, 'ml_stacking_and_conformal')
plt.show()

### 8.7 Model selection by 5-fold cross-validation
The same eight models, now compared by 5-fold cross-validation on the training data only, so that the test data play
no part in choosing the model.

In [ ]:
kf = KFold(5, shuffle=True, random_state=RS)
rows = []
for name in MODEL_NAMES:
    model = ml_model(name)
    r2 = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2', n_jobs=1)
    mae = -cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', n_jobs=1)
    rows.append({'model': name, 'cv_R2': r2.mean(), 'cv_R2_sd': r2.std(), 'cv_MAE': mae.mean()})
cv_table = pd.DataFrame(rows).sort_values('cv_R2', ascending=False)
cv_table.to_csv(os.path.join(ML_DIR, 'cv5_model_selection.csv'), index=False)
print(cv_table.round(4).to_string(index=False))

### 8.8 Validation without information leaking from training to test data
Optimizers sample good regions repeatedly, so some trials are exact copies of others. The model is therefore also tested
after removing duplicates, with each seed left out in turn (leave-one-seed-out), and with all GA trials held out.

In [ ]:
key = pd.Series(['|'.join(f'{v:.10g}' for v in row) for row in data[VARS].values])
in_train = set(key.iloc[idx_train])
dup = np.array([k in in_train for k in key.iloc[idx_test]])
print(f'Test trials with an exact copy in the training data: {dup.sum()} ({dup.mean() * 100:.1f}%)\n')

validation = {'Random 80/20 split': metrics(y_test, extra_trees().fit(X_train, y_train).predict(X_test))}

unique = data.loc[~key.duplicated()].reset_index(drop=True)
a, b, c, d = train_test_split(log_X(unique), unique['eta'].values, test_size=0.2, random_state=RS)
validation['Duplicates removed, then random split'] = metrics(d, extra_trees().fit(a, c).predict(b))

loso = np.zeros_like(y)
for tr, te in GroupKFold(n_splits=6).split(X, y, seeds):
    loso[te] = extra_trees().fit(X[tr], y[tr]).predict(X[te])
validation['Leave-one-seed-out'] = metrics(y, loso)

is_ga = (data.algorithm == 'GA').values
train_mask = ~is_ga & ~key.isin(set(key[is_ga])).values
validation['GA trials held out'] = metrics(y[is_ga], extra_trees().fit(X[train_mask], y[train_mask]).predict(X[is_ga]))

for label, m in validation.items():
    show(label, m)

### 8.9 Test on independent devices
The Monte Carlo devices of step 7.1 were simulated separately and were never used for training.
The model is trained on all trials and then predicts their PCE.

In [ ]:
mc_devices = MC.rename(columns={'WO3_thickness': 'WO3_t', 'MASnIBr2_thickness': 'MASnIBr2_t', 'CBTS_thickness': 'CBTS_t'})
pred = extra_trees().fit(X, y).predict(log_X(mc_devices))
print(f"{len(mc_devices)} Monte Carlo devices, PCE {mc_devices.eta.min():.2f}–{mc_devices.eta.max():.2f}%, "
      f"mean error {np.mean(pred - mc_devices.eta.values):+.4f} percentage points")
show('Monte Carlo devices', metrics(mc_devices.eta.values, pred))

### 8.10 Direct versus stacked model
Is the stacked model really better than the direct model? Paired bootstrap of the test data (2000 resamples) and
leave-one-seed-out (Extra Trees with 300 trees, the functions of step 8.3).

In [ ]:
p_dir, _, p_stack = stacked_fit_predict(extra_trees(300), X_train, y_train, Y_train, X_test, n_trees=300)
rng = np.random.default_rng(0)
dR2, dMAE = [], []
for _ in range(2000):
    i = rng.integers(0, len(y_test), len(y_test))
    dR2.append(r2_score(y_test[i], p_stack[i]) - r2_score(y_test[i], p_dir[i]))
    dMAE.append(mean_absolute_error(y_test[i], p_stack[i]) - mean_absolute_error(y_test[i], p_dir[i]))
print('Stacked minus direct (random split, 2000 bootstrap resamples):')
print(f'  ΔR²  = {np.mean(dR2):+.4f}, 95% interval {np.percentile(dR2, 2.5):+.4f} to {np.percentile(dR2, 97.5):+.4f}')
print(f'  ΔMAE = {np.mean(dMAE):+.4f}, 95% interval {np.percentile(dMAE, 2.5):+.4f} to {np.percentile(dMAE, 97.5):+.4f}')

loso_dir, loso_stack = np.zeros_like(y), np.zeros_like(y)
for tr, te in GroupKFold(n_splits=6).split(X, y, seeds):
    loso_dir[te], _, loso_stack[te] = stacked_fit_predict(extra_trees(300), X[tr], y[tr], Y[tr], X[te], n_trees=300)
print('\nLeave-one-seed-out:')
show('Direct PCE (Extra Trees)', metrics(y, loso_dir))
show('Stacked model', metrics(y, loso_stack))

### 8.11 Conformal coverage in different PCE regions
A single interval width can be too wide for some PCE regions and too narrow for others. Coverage is therefore checked per
PCE region, and a region-wise (Mondrian) interval with the finite-sample quantile is compared with the single interval.

In [ ]:
BINS = [0, 23, 24, 25, 25.2, 30]
REGIONS = ['< 23%', '23–24%', '24–25%', '25–25.2%', '≥ 25.2%']

def q_finite(residuals, alpha=0.1):
    n = len(residuals)
    return np.quantile(residuals, min(math.ceil((n + 1) * (1 - alpha)) / n, 1.0), method='higher')

cp_model = extra_trees().fit(X_fit, y_fit)
p_cal, p_test = cp_model.predict(X_cal), cp_model.predict(X_test)
r_cal = np.abs(y_cal - p_cal)
del cp_model

q_global = np.quantile(r_cal, 0.9)
region_cal = pd.cut(p_cal, BINS, labels=REGIONS, right=False)
region_test = pd.cut(p_test, BINS, labels=REGIONS, right=False)
q_region = {g: q_finite(r_cal[np.asarray(region_cal == g)]) for g in REGIONS}
q_test = np.array([q_region[g] for g in region_test])
cover = pd.DataFrame({'region': region_test, 'global': np.abs(y_test - p_test) <= q_global,
                      'mondrian': np.abs(y_test - p_test) <= q_test})

print(f'Single interval ±{q_global:.4f}: coverage {cover["global"].mean() * 100:.1f}%; '
      f'region-wise intervals: coverage {cover["mondrian"].mean() * 100:.1f}%\n')
print(f"{'PCE region':<10} {'single interval':>16} {'region half-width':>18} {'region-wise':>12}")
for g in REGIONS:
    s = cover[cover.region == g]
    print(f"{g:<10} {s['global'].mean() * 100:15.1f}% {'±%.4f' % q_region[g]:>18} {s['mondrian'].mean() * 100:11.1f}%")

cover_loso = np.zeros(len(y), bool)
for tr, te in GroupKFold(6).split(X, y, seeds):
    cal_seed = np.unique(seeds[tr])[0]
    cal, fit = tr[seeds[tr] == cal_seed], tr[seeds[tr] != cal_seed]
    m = extra_trees(300).fit(X[fit], y[fit])
    cover_loso[te] = np.abs(y[te] - m.predict(X[te])) <= q_finite(np.abs(y[cal] - m.predict(X[cal])))
del m
print(f'\nLeave-one-seed-out coverage (one training seed used for calibration): {cover_loso.mean() * 100:.1f}%')

## Part 9 — Summary

### 9.1 Key numbers
The main results are collected in `output/summary.json`.

In [ ]:
SUMMARY = {
    'device': 'FTO/WO3/MASnIBr2/CBTS/Au',
    'final_device': FINAL,
    'best_pce_per_seed': {a: BEST_PER_SEED[a].round(4).tolist() for a in ALGOS},
    'mean_sd_per_algorithm': {a: [float(desc.loc[a, 'mean']), float(desc.loc[a, 'sd'])] for a in ALGOS},
    'run_time_h': hours,
    'loss_decomposition': LOSSES,
    'shockley_queisser': {'limit_at_1.75_eV': SQ, 'final_device_share_pct': SHARE_OF_SQ},
    'thickness_grid': GRID_SUMMARY,
    'monte_carlo': MC_STATS,
    'ml_best_model': {'name': BEST_NAME, **MODEL_TABLE.set_index('model').loc[BEST_NAME, ['test_r2', 'test_mae']].to_dict()},
}
with open(os.path.join(OUT_DIR, 'summary.json'), 'w') as f:
    json.dump(SUMMARY, f, indent=2, default=float)

print(f"Final device ({FINAL['algorithm']}, seed {FINAL['seed']}): PCE {FINAL['PCE']:.4f}%, "
      f"Voc {FINAL['Voc']:.4f} V, Jsc {FINAL['Jsc']:.4f} mA/cm2, FF {FINAL['FF']:.4f}%")
print(f"Thickness WO3 / MASnIBr2 / CBTS: {FINAL['WO3_t']*1000:.0f} / {FINAL['MASnIBr2_t']*1000:.0f} / {FINAL['CBTS_t']*1000:.0f} nm")
for a in ALGOS:
    print(f"{a:<5} best PCE {desc.loc[a, 'mean']:.4f} ± {desc.loc[a, 'sd']:.4f}% over six seeds, {hours[a]:.1f} h")
print(f"Monte Carlo: {MC_STATS['eta_mean']:.3f} ± {MC_STATS['eta_std']:.3f}% ({MC_STATS['n_success']} devices)")
print(f"Share of the Shockley–Queisser limit: {SHARE_OF_SQ['PCE']:.1f}%")
print(f"ML model: {BEST_NAME}, test R² {best_row.test_r2:.4f}, MAE {best_row.test_mae:.4f} percentage points")
print('\nAll results are in', OUT_DIR)